# GREI / DataCite outputs cited in Web of Science — cited-reference search

Finds Web of Science papers whose **reference lists** cite a DOI from `data/GREI-DataCite-NIHOnly.csv` (from Keith Herzog
and Veronica Hoyo), saves who cited what, and adds WoS citation counts to every row of that file.

## ▶ How to run — the same way every time
1. **Settings (section 1):** check the paths. `TEST_MODE = True` for a quick trial on ~5,000 records; `False` for the real run.
2. **Run → Run All Cells** (or *Kernel → Restart Kernel and Run All Cells*). Never pick individual cells: every cell is
   meant to run every time, and each one decides for itself what to do from `TEST_MODE`.
3. **Read the last section, "5. What happened and what to do next".** It tells you the next step in plain words.

| | `TEST_MODE = True` (trial) | `TEST_MODE = False` (real run) |
|---|---|---|
| Section 3a — Test | reads ~5,000 records, runs checks | skipped |
| Section 3b — Full run | skipped | processes the whole new export |
| Files written | `TEST_…` copies only; the export stays "new" | real results; the export is marked done |

**Each new export:** download `wos-data.zip` to Downloads → Run All (`TEST_MODE = False`) → when section 5 says it is done,
delete `wos-data.zip` → download the next one. Results build up in `GREI_WoS_outputs/` (keep that folder); nothing is
counted twice, even though every export has the same name.

Only section 1 needs editing. Sections 2–5 run automatically; their long tables are hidden unless `SHOW_DETAILS = True`.

## 1. Settings ✏️ — the only cell you edit

In [ ]:
# =============================  CONFIGURATION  =============================
from pathlib import Path
import os

# --- Where things are -------------------------------------------------------
# Project folder = the folder this notebook is in (Jupyter starts there). Everything below is relative to it.
BASE_DIR = Path(os.environ.get("GREI_BASE_DIR", Path.cwd()))

# Target DOIs (from Keith Herzog and Veronica Hoyo). CSV or Excel; a path relative to BASE_DIR, or a full path.
TARGET_FILE       = "data/GREI-DataCite-NIHOnly.csv"
TARGET_SHEET      = 0                              # only used if TARGET_FILE is an Excel file
TARGET_DOI_COLUMN = "id"                           # AUTHORITATIVE DOI column used for matching
QA_DOI_COLUMN     = "attributes_doi"               # only used for a QA comparison; never used for matching

# Where the WoS export ZIP is downloaded. Only files whose names match WOS_FILE_PATTERN are used, so other
# ZIPs in Downloads are ignored. Processed archives are remembered by a CONTENT fingerprint, so deleting
# wos-data.zip after processing and downloading the next export (also called wos-data.zip) is safe.
WOS_DATA_DIR           = Path.home() / "Downloads"
WOS_FILE_PATTERN       = "wos-data*.zip"     # matches wos-data.zip, wos-data (1).zip, ... (case-insensitive)
WOS_SEARCH_SUBFOLDERS  = False               # False = only the top level of WOS_DATA_DIR
ZIP_ONLY     = True          # True = only .zip exports are inputs; False = also loose .xml/.xml.gz/.jsonl files
ZIP_XML_MEMBER_SUFFIX = ".xml"   # inside a ZIP only members ending with this are read; each = one WoS record.
                                 # Other entries (manifests, readme, ...) are counted and ignored.

# TDM Studio wraps every WoS record in ProQuest XML (<RECORD> with <GOID>, <Obj>, <PublicationInfo> ...) and stores
# the ORIGINAL Clarivate WoS record (<REC>, including the cited-reference list) as text in the element below.
# The notebook detects this automatically and unwraps it; these settings only need changing if ProQuest moves it.
PROQUEST_WRAPPER_TAG  = "RECORD"
EMBEDDED_WOS_XML_PATH = "Obj/Notes/Note/RawFilename"

# All results and the processing log go here (created if missing). Keep this folder between batches: it is the
# memory of what has been processed. It is deliberately NOT inside Downloads.
OUTPUT_DIR = BASE_DIR / "GREI_WoS_outputs"

# --- Which WoS files to process ---------------------------------------------
PROCESS_FILES  = "all"       # "all"  or  "selected"
SELECTED_FILES = []          # used when PROCESS_FILES = "selected"; file names, relative paths, or
                             # wildcard patterns, e.g. ["wos_2019/*", "part-0001.xml.gz"]

SKIP_COMPLETED_FILES  = True # skip files the log says were completed successfully
FORCE_REPROCESS_FILES = []   # names / relative paths / patterns to reprocess even if completed

# --- Test mode ----------------------------------------------------------------
TEST_MODE               = False   # True = small trial run only (writes TEST_ files, no log entries)
TEST_MAX_RECORDS        = 5_000   # max WoS records read in test mode (first validation run)
TEST_MAX_FILES          = 1       # test mode uses ONE ZIP (an unprocessed one if available)
TEST_STOP_AFTER_MATCHES = 25      # stop a test file early once this many relationships are found

# --- Checkpointing / progress -------------------------------------------------
CHECKPOINT_EVERY_RECORDS = 100_000  # write found matches to disk at least this often inside a file
PROGRESS_EVERY_RECORDS   = 250_000  # progress message interval (used when tqdm is not installed)
VERBOSE                  = True
SHOW_DETAILS             = False    # True = also show long diagnostic tables (column lists, full record structure)
STOP_ON_FILE_ERROR       = False    # False = log the error and continue with the next file
IGNORE_STATE_WARNINGS    = False    # True = run even if the start-of-run integrity check finds missing results

# --- Matching options -----------------------------------------------------------
# Also look for DOIs written anywhere in a cited reference's text (e.g. "DOI 10.5061/dryad.x" in an
# article-number field), not only in its dedicated DOI field. Matching stays EXACT after normalization.
SCAN_ALL_REFERENCE_TEXT_FOR_DOIS = True

# Speed-up for the full run (about 10x): a record can only match if its text contains one of your targets' DOI
# prefixes (e.g. "10.5281/" Zenodo, "10.5061/" Dryad, "10.17605/" OSF). Records without any are counted and their
# ID/date recorded, but not fully parsed. Results are identical. Test mode always reads every record in full.
FAST_PREFILTER = True

# --- Inspection / advanced ------------------------------------------------------
INSPECT_SAMPLE_RECORDS = 200       # records read from the sample file to profile the schema
XML_MODE               = "auto"    # "auto", "stream" (one big XML document), or "lines" (one record per line)
MAX_JSON_FILE_MB       = 500       # single-document .json files larger than this are refused (use JSONL)

# --- Generic TDM Studio exports ("wos-data.zip", "wos-data (1).zip", ...) ----------------------
# Files ending with these suffixes are unfinished transfers. They are reported and NEVER processed.
INCOMPLETE_TRANSFER_SUFFIXES = [".part", ".crdownload", ".partial", ".download"]

# Your TDM Studio datasets and the publication-date range each was built for (inclusive).
# Used only to SUGGEST which generic ZIP is which. Add/remove rows freely.
#   (dataset name,      start date,   end date,     expected record count or None)
EXPECTED_DATASETS = [
    ("GREIWOS20220104", "2022-01-01", "2022-04-30", 1548845),   # Jan-Apr 2022
    ("GREIWOS20220508", "2022-05-01", "2022-08-31", 1015723),   # May-Aug 2022
    ("GREIWOS20220912", "2022-09-01", "2022-12-31", 1121265),   # Sep-Dec 2022
    ("GREIWOS20230104", "2023-01-01", "2023-04-30", 1513243),   # Jan-Apr 2023
    ("GREIWOS20230508", "2023-05-01", "2023-08-31", 978200),   # May-Aug 2023
    ("GREIWOS20230912", "2023-09-01", "2023-12-31", 1088799),   # Sep-Dec 2023
    ("GREIWOS20240104", "2024-01-01", "2024-04-30", None),   # Jan-Apr 2024
    ("GREIWOS20240508", "2024-05-01", "2024-08-31", None),   # May-Aug 2024
    ("GREIWOS20240912", "2024-09-01", "2024-12-31", None),   # Sep-Dec 2024
    ("GREIWOS20250104", "2025-01-01", "2025-04-30", None),   # Jan-Apr 2025
    ("GREIWOS20250508", "2025-05-01", "2025-08-31", None),   # May-Aug 2025
    ("GREIWOS20250912", "2025-09-01", "2025-12-31", None),   # Sep-Dec 2025
    ("GREIWOS20260104", "2026-01-01", "2026-04-30", None),   # Jan-Apr 2026
    ("GREIWOS20260508", "2026-05-01", "2026-08-31", None),   # May-Aug 2026
    ("GREIWOS20260912", "2026-09-01", "2026-12-31", None),   # Sep-Dec 2026
]

# Optional: record which archive is which once you know (most reliable provenance).
# Key = the first 12+ characters of the archive's content fingerprint (column "fingerprint" in the status
# table / manifest). Because every export downloads as "wos-data.zip", use the FINGERPRINT, not the file name:
# a file-name key is ignored as soon as two different archives have had that name. You can instead type names into
# the manual_dataset_name / manual_expected_start / manual_expected_end columns of
# GREI_WoS_archive_manifest.csv — entries here take precedence over the CSV.
ARCHIVE_ASSIGNMENTS = {}      # e.g. {"74b9835a850f": "GREIWOS20230912"} - optional

PRE_INVENTORY_MODE = "sample"       # before processing, profile new archives: "sample" (fast), "full", or "off"
INVENTORY_SAMPLE_RECORDS = 50_000  # approx. records read per archive in "sample" mode, spread across its members
ASSIGN_MIN_SHARE_INSIDE  = 0.90     # suggest a dataset only if ≥90% of dated records fall in its range ...
ASSIGN_MIN_MARGIN        = 0.50     # ... and it beats the runner-up by ≥50 points; otherwise "ambiguous"
COVERAGE_MONTH_MIN_SHARE = 0.01     # a month is "covered" by an archive if it holds ≥1% of that archive's records

COMPUTE_FULL_FILE_SHA256 = False  # True = also record a full SHA-256 of each ZIP (reads the whole file once more)
KEEP_STATE_BACKUPS       = 5      # copies of the small state files kept in OUTPUT_DIR/state_backups (0 = off)

RENAME_ARCHIVES       = False       # True = rename identified archives (e.g. to GREIWOS20230508.zip)
RENAME_REQUIRE_MANUAL = True        # True = only rename archives whose dataset YOU assigned
# ============================================================================

## 2. Automatic checks — nothing to edit from here on
### 2a. Setup (packages, output folder)
Nothing to edit here. `lxml` and `tqdm` are used if installed; otherwise the standard library is used.

In [ ]:
import re, sys, csv, json, gzip, zipfile, io, time, copy, math, ast, fnmatch, hashlib, traceback, shutil, html
from collections import Counter, defaultdict
from datetime import datetime, date, timedelta
from array import array
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

try:
    from lxml import etree as LET
    HAVE_LXML = True
except ImportError:
    HAVE_LXML = False

try:
    from tqdm.auto import tqdm
    HAVE_TQDM = True
except ImportError:
    HAVE_TQDM = False

try:
    import openpyxl
    from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
    HAVE_OPENPYXL = True
except ImportError:
    HAVE_OPENPYXL = False
    ILLEGAL_CHARACTERS_RE = re.compile(r"[\000-\010]|[\013-\014]|[\016-\037]")

try:
    display                      # available inside Jupyter
except NameError:                # fallback if run as a plain script
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

csv.field_size_limit(min(sys.maxsize, 2**31 - 1))
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)

print(f"Python      : {sys.version.split()[0]}")
print(f"pandas      : {pd.__version__}")
print(f"lxml        : {'yes ' + LET.__version__ if HAVE_LXML else 'NOT installed -> using xml.etree (slower but fine)'}")
print(f"tqdm        : {'yes' if HAVE_TQDM else 'NOT installed -> printed progress messages instead'}")
print(f"openpyxl    : {'yes' if HAVE_OPENPYXL else 'NOT installed -> Excel output disabled; run  pip install openpyxl'}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR = OUTPUT_DIR / "summary_tables"
SUMMARY_DIR.mkdir(exist_ok=True)
ARCHIVE_DIR = OUTPUT_DIR / "archive_inventory"      # per-archive date profiles + hashed UT lists
ARCHIVE_DIR.mkdir(exist_ok=True)
COVERAGE_DIR = OUTPUT_DIR / "coverage_report"
COVERAGE_DIR.mkdir(exist_ok=True)
MANIFEST_CSV   = OUTPUT_DIR / "GREI_WoS_archive_manifest.csv"
RENAME_LOG_CSV = OUTPUT_DIR / "GREI_WoS_archive_rename_log.csv"
MANIFEST_LOOKUP = {}                                 # fingerprint -> manifest row (filled in section 2g)

def output_paths(test=False):
    """All output file locations. Test runs get a TEST_ prefix so they never mix with real results."""
    p = "TEST_" if test else ""
    return {
        "log":          OUTPUT_DIR / "GREI_WoS_processing_log.csv",            # never written in test mode
        "edges_raw":    OUTPUT_DIR / f"{p}GREI_WoS_citation_edges_raw_appendonly.csv",
        "master_csv":   OUTPUT_DIR / f"{p}GREI_WoS_citation_edges_master.csv",
        "master_xlsx":  OUTPUT_DIR / f"{p}GREI_WoS_citation_edges_master.xlsx",
        "malformed":    OUTPUT_DIR / f"{p}GREI_WoS_malformed_cited_ref_doi_samples.csv",
        "nearmiss":     OUTPUT_DIR / f"{p}GREI_WoS_possible_version_near_misses.csv",
        "agg":          OUTPUT_DIR / f"{p}GREI_WoS_target_aggregates.csv",
        "merged_xlsx":  OUTPUT_DIR / f"{p}GREI-DataCite-NIHOnly_with_WoS_citations.xlsx",
        "merged_csv":   OUTPUT_DIR / f"{p}GREI-DataCite-NIHOnly_with_WoS_citations.csv",
        "compare":      OUTPUT_DIR / f"{p}GREI_WoS_vs_DataCite_citation_comparison.csv",
        "unmatched":    OUTPUT_DIR / f"{p}GREI_WoS_unmatched_target_dois.csv",
        "summary_prefix": p,
    }

def now_iso():
    return datetime.now().astimezone().isoformat(timespec="seconds")

def human_size(n):
    n = float(n or 0)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024 or unit == "TB":
            return f"{n:,.1f} {unit}"
        n /= 1024

def warn(msg):
    print(f"⚠️  WARNING: {msg}")

MULTI_SEP = " | "     # separator for multi-valued fields (authors, organizations, countries, ...)
ADDR_SEP  = " || "    # separator between full raw address strings

def _split_multi_set(v):
    return {x.strip() for x in str(v).split(MULTI_SEP) if x.strip()}

# ---- CSV / log helpers ------------------------------------------------------------------------
LOG_COLUMNS = ["file_key", "file_name", "dataset", "content_fingerprint", "file_size_bytes", "run_id", "status",
               "started_at", "finished_at", "seconds", "xml_members", "xml_members_read", "record_count_check",
               "records_examined", "records_prefiltered", "cited_refs_examined",
               "cited_refs_with_doi", "raw_reference_matches", "unique_relationships", "unique_target_dois_matched",
               "records_missing_ut", "malformed_ref_dois", "version_near_misses", "warnings", "error", "settings"]

def _csv_header(path):
    with open(path, newline="", encoding="utf-8") as f:
        return next(csv.reader(f), [])

def ensure_csv_columns(path, columns):
    """Add columns introduced by a newer notebook version to an existing CSV, keeping all its rows."""
    path = Path(path)
    if not path.exists() or set(columns) <= set(_csv_header(path)):
        return
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    for c in columns:
        if c not in df.columns:
            df[c] = ""
    df = df[list(columns) + [c for c in df.columns if c not in columns]]
    atomic_to_csv(df, path)
    print(f"ℹ️  Upgraded {path.name} with new column(s).")

def append_csv(path, rows, columns):
    if not rows:
        return
    exists = Path(path).exists()
    fields = _csv_header(path) if exists else list(columns)
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        if not exists:
            w.writeheader()
        w.writerows(rows)
        f.flush()
        os.fsync(f.fileno())

def _replace_with_retry(tmp, path):
    """os.replace, but if the destination is open in Excel (Windows locks it) wait and retry, then save the
    result next to it with a time stamp for report files; state files raise a clear error instead."""
    for attempt in range(5):
        try:
            os.replace(tmp, path)
            return path
        except PermissionError:
            time.sleep(2)
    state = {output_paths(False)["master_csv"].name, output_paths(False)["log"].name, "GREI_WoS_archive_manifest.csv"}
    if Path(path).name in state:
        raise PermissionError(f"Cannot update {Path(path).name} - is it open in Excel? Close it and re-run "
                              f"(nothing was lost: the new version is saved as {Path(tmp).name}).")
    alt = Path(path).with_name(f"{Path(path).stem}_{datetime.now():%Y%m%d_%H%M%S}{Path(path).suffix}")
    os.replace(tmp, alt)
    warn(f"{Path(path).name} is open in another program; saved this version as {alt.name} instead.")
    return alt

def atomic_to_csv(df, path):
    tmp = Path(str(path) + ".tmp")
    df.to_csv(tmp, index=False, encoding="utf-8")
    return _replace_with_retry(tmp, path)

def read_log():
    p = output_paths(False)["log"]
    if not p.exists():
        return pd.DataFrame(columns=LOG_COLUMNS)
    log = pd.read_csv(p, dtype=str, keep_default_na=False)
    for c in LOG_COLUMNS:
        if c not in log.columns:
            log[c] = ""
    return log

ensure_csv_columns(output_paths(False)["log"], LOG_COLUMNS)
print("\nOutputs will be written to:", OUTPUT_DIR)

### 2b. Find the target file and the WoS export

In [ ]:
print("Current working directory:", Path.cwd())
print("BASE_DIR:", BASE_DIR, "| exists:", BASE_DIR.exists())

if BASE_DIR.exists():
    rows = []
    for p in sorted(BASE_DIR.iterdir(), key=lambda x: x.name.lower()):
        try:
            st = p.stat()
        except OSError:
            continue
        rows.append({"name": p.name,
                     "type": "folder" if p.is_dir() else "file",
                     "size": "" if p.is_dir() else human_size(st.st_size),
                     "modified": datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M")})
    display(pd.DataFrame(rows))
else:
    warn("BASE_DIR does not exist. Edit BASE_DIR in the configuration cell.")

In [ ]:
def locate_target_file(name):
    p = Path(name)
    if p.is_absolute():
        return p if p.exists() else None
    for c in [BASE_DIR / name, Path.cwd() / name]:
        if c.exists():
            return c
    hits = []
    if BASE_DIR.exists():
        for dirpath, dirnames, filenames in os.walk(BASE_DIR):
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if name in filenames:
                hits.append(Path(dirpath) / name)
    hits = [h for h in hits if OUTPUT_DIR not in h.parents]
    if len(hits) > 1:
        warn(f"Several copies of {name} found; using the first:\n   " + "\n   ".join(map(str, hits)))
    return hits[0] if hits else None

TARGET_PATH = locate_target_file(TARGET_FILE)
if TARGET_PATH is None:
    raise FileNotFoundError(f"Could not find {TARGET_FILE!r} under {BASE_DIR}. Put it in the data folder next to this notebook, or set TARGET_FILE to its full path.")
print("Target workbook:", TARGET_PATH, "|", human_size(TARGET_PATH.stat().st_size))

In [ ]:
# ---- Find candidate Web of Science files -------------------------------------------------
KNOWN_EXTS = [".xml.gz", ".jsonl.gz", ".json.gz", ".xml", ".jsonl", ".json", ".zip", ".gz"]

def file_kind(name):
    n = str(name).lower()
    for ext in KNOWN_EXTS:
        if n.endswith(ext):
            return ext
    return None

def data_format_for(name):
    """'xml', 'jsonl' or 'json' for a file/member name (compression suffix ignored)."""
    n = str(name).lower()
    if n.endswith(".gz"):
        n = n[:-3]
    if n.endswith(".jsonl") or n.endswith(".ndjson"):
        return "jsonl"
    if n.endswith(".json"):
        return "json"
    return "xml"     # .xml, or an unlabelled .gz -> assume XML (checked by the peek below)

WOS_ROOT = Path(WOS_DATA_DIR).expanduser() if WOS_DATA_DIR is not None else BASE_DIR
if not WOS_ROOT.exists():
    raise FileNotFoundError(f"WOS_DATA_DIR does not exist: {WOS_ROOT}. Edit WOS_DATA_DIR in the configuration cell.")
print(f"Looking for {WOS_FILE_PATTERN!r} in: {WOS_ROOT}" + (" (and sub-folders)" if WOS_SEARCH_SUBFOLDERS else ""))

def is_incomplete_transfer(name):
    low = str(name).lower()
    return any(low.endswith(sfx) for sfx in INCOMPLETE_TRANSFER_SUFFIXES)

def _name_matches(name):
    return fnmatch.fnmatch(name.lower(), WOS_FILE_PATTERN.lower()) if WOS_FILE_PATTERN else True

def _strip_transfer_suffix(name):
    low = name.lower()
    for sfx in INCOMPLETE_TRANSFER_SUFFIXES:
        if low.endswith(sfx):
            return name[: -len(sfx)]
    return name

def discover_candidate_files(root):
    """Return (complete candidate files, incomplete transfers). Incomplete transfers are never processed.
    A transfer can be a file (wos-data.zip.crdownload, .part) or, with Safari, a folder (wos-data.zip.download)."""
    out, incomplete = [], []
    out_dir = OUTPUT_DIR.resolve()
    for dirpath, dirnames, filenames in os.walk(root):
        keep = []
        for d in sorted(dirnames):
            if d.startswith(".") or (Path(dirpath) / d).resolve() == out_dir:
                continue
            if is_incomplete_transfer(d) and _name_matches(_strip_transfer_suffix(d)):
                incomplete.append(Path(dirpath) / d)
            elif WOS_SEARCH_SUBFOLDERS:
                keep.append(d)
        dirnames[:] = keep
        for fn in sorted(filenames):
            if fn.startswith(".") or fn.startswith("~$"):
                continue
            if is_incomplete_transfer(fn):
                if _name_matches(_strip_transfer_suffix(fn)):
                    incomplete.append(Path(dirpath) / fn)
            elif _name_matches(fn) and file_kind(fn) and (not ZIP_ONLY or fn.lower().endswith(".zip")):
                out.append(Path(dirpath) / fn)
    return out, incomplete

def _path_size(p):
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) if p.is_dir() else p.stat().st_size

def archive_facts(path):
    """Content fingerprint + member counts WITHOUT reading the data.
    ZIP: built from the central directory (member names, CRC-32 checksums, sizes), so an identical
    re-download gets the same fingerprint whatever its file name. Other files: size + first/last 4 MB."""
    p = Path(path)
    size = p.stat().st_size
    h = hashlib.sha256(f"size={size}\n".encode())
    if p.name.lower().endswith(".zip"):
        try:
            with zipfile.ZipFile(p) as zf:
                infos = [i for i in zf.infolist() if not i.is_dir()]
        except (zipfile.BadZipFile, OSError, EOFError) as e:
            return {"content_fingerprint": "", "fingerprint_method": "", "data_members": 0, "xml_members": 0,
                    "non_xml_members": 0, "non_xml_member_examples": "",
                    "problem": f"unreadable ZIP ({str(e)[:80]}) - probably an incomplete transfer or a corrupt file"}
        for i in sorted(infos, key=lambda i: i.filename):
            h.update(f"{i.filename}|{i.CRC}|{i.file_size}\n".encode())
        xml = [i for i in infos if is_zip_record_member(i)]
        other = [i.filename for i in infos if not is_zip_record_member(i)]
        return {"content_fingerprint": h.hexdigest(),
                "fingerprint_method": "ZIP directory (member names, CRC-32, sizes)",
                "data_members": len(xml), "xml_members": len(xml),
                "non_xml_members": len(other), "non_xml_member_examples": MULTI_SEP.join(other[:5]),
                "problem": "" if xml else f"ZIP contains no {ZIP_XML_MEMBER_SUFFIX} members"}
    with open(p, "rb") as f:
        h.update(f.read(4 << 20))
        if size > (8 << 20):
            f.seek(size - (4 << 20))
            h.update(f.read())
    return {"content_fingerprint": h.hexdigest(), "fingerprint_method": "size + first/last 4 MB",
            "data_members": 1, "xml_members": 0, "non_xml_members": 0, "non_xml_member_examples": "",
            "problem": ""}

def peek_bytes(path, n=65536):
    """Return the first n *decompressed* bytes of a file (first member for .zip). Reads nothing else."""
    try:
        if str(path).lower().endswith(".zip"):
            with zipfile.ZipFile(path) as zf:
                first_xml = next((i for i in zf.infolist() if is_zip_record_member(i)), None)
                if first_xml is None:
                    return b""
                with zf.open(first_xml) as f:
                    return f.read(n)
        opener = gzip.open if str(path).lower().endswith(".gz") else open
        with opener(path, "rb") as f:
            return f.read(n)
    except Exception as e:
        return f"<<could not read: {e}>>".encode()

def is_zip_record_member(info):
    """A ZIP entry that holds WoS record XML (default: name ends with .xml)."""
    return not info.is_dir() and info.filename.lower().endswith(ZIP_XML_MEMBER_SUFFIX.lower())

WOS_MARKERS = [b"<REC", b":REC", b"wok5", b"static_data", b"fullrecord_metadata", b"\"UID\"", b"WOS:"]

def looks_like_wos(head):
    return sum(m in head for m in WOS_MARKERS) >= 2

candidates, INCOMPLETE_FILES = discover_candidate_files(WOS_ROOT) if WOS_ROOT.exists() else ([], [])
if not candidates and not INCOMPLETE_FILES:
    other_zips = sorted(p.name for p in WOS_ROOT.iterdir() if p.is_file() and p.name.lower().endswith(".zip"))
    print(f"ℹ️  No file matching {WOS_FILE_PATTERN!r} in {WOS_ROOT}.")
    if other_zips:
        print("   ZIP files that ARE there (not used): " + ", ".join(other_zips[:10])
              + "\n   If one is the WoS export, rename it to wos-data.zip or change WOS_FILE_PATTERN.")
    wos_dirs = sorted(p.name for p in WOS_ROOT.iterdir() if p.is_dir() and _name_matches(p.name + ".zip"))
    if wos_dirs:
        print(f"   Found a FOLDER named {wos_dirs[0]!r}: the browser may have unzipped the export automatically. "
              "Re-download it as a .zip (e.g. turn off Safari's 'Open safe files after downloading').")

if INCOMPLETE_FILES:
    print("⏳ INCOMPLETE TRANSFERS — these files will NOT be processed:")
    display(pd.DataFrame([{
        "file": str(p.relative_to(BASE_DIR) if BASE_DIR in p.parents else p),
        "size": human_size(_path_size(p)),
        "last_modified": datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M"),
        "status": "appears to be an incomplete transfer (still downloading or interrupted)"}
        for p in INCOMPLETE_FILES]))
    print("   Wait until the download finishes (the suffix disappears), then re-run the notebook.\n")
else:
    print("No incomplete transfers (.part etc.) found.")

rows = []
for p in candidates:
    head = peek_bytes(p, 65536)
    rel = p.relative_to(WOS_ROOT)
    facts = archive_facts(p)
    rows.append({
        "file_key": rel.as_posix(),
        "dataset": rel.parts[0] if len(rel.parts) > 1 else WOS_ROOT.name,
        "file_name": p.name,
        "extension": file_kind(p.name),
        "size_bytes": p.stat().st_size,
        "size": human_size(p.stat().st_size),
        "likely_wos": looks_like_wos(head) and not facts["problem"],
        **facts,
        "fingerprint_short": facts["content_fingerprint"][:12],
        "path": str(p),
    })
FILES_DF = pd.DataFrame(rows, columns=["file_key", "dataset", "file_name", "extension", "size_bytes", "size",
                                       "likely_wos", "content_fingerprint", "fingerprint_method", "data_members",
                                       "xml_members", "non_xml_members", "non_xml_member_examples", "problem",
                                       "fingerprint_short", "path"])

print(f"\nCandidate data files found: {len(FILES_DF):,}")
if len(FILES_DF):
    print("\nBy extension:")
    display(FILES_DF.groupby("extension").agg(files=("file_key", "count"), total_bytes=("size_bytes", "sum"))
            .assign(total_size=lambda d: d.total_bytes.map(human_size)).drop(columns="total_bytes"))
    print("\nFirst 40 candidate files:")
    display(FILES_DF.drop(columns=["path", "size_bytes", "content_fingerprint", "fingerprint_method"]).head(40))

bad = FILES_DF[FILES_DF.problem != ""]
if len(bad):
    print("\n❌ Files that cannot be read and will NOT be processed:")
    display(bad[["file_key", "size", "problem"]])
FILES_DF["likely_wos"] = FILES_DF.likely_wos.astype(bool)    # (also correct when no files are present)
WOS_FILES_DF = FILES_DF.loc[FILES_DF.likely_wos].reset_index(drop=True)
not_wos = FILES_DF.loc[~FILES_DF.likely_wos & (FILES_DF.problem == "")]
if len(not_wos):
    print(f"\n{len(not_wos)} candidate file(s) did NOT look like WoS and will be ignored, e.g.:")
    display(not_wos[["file_key", "size"]].head(10))
same = WOS_FILES_DF[WOS_FILES_DF.content_fingerprint.duplicated(keep=False)]
if len(same):
    warn("Some files have IDENTICAL content (duplicate downloads). Only one copy of each will be processed:")
    display(same[["file_key", "size", "fingerprint_short"]].sort_values("fingerprint_short"))
if WOS_FILES_DF.empty:
    warn("No WoS-looking files were found. Check WOS_DATA_DIR / that your TDM Studio exports are present.")
else:
    print(f"\n✅ {len(WOS_FILES_DF):,} likely WoS file(s), total {human_size(WOS_FILES_DF.size_bytes.sum())}")

In [ ]:
# ---- Peek at the very beginning of one WoS file (a few KB only) ---------------------------
_XML_MODE_CACHE = {}

def detect_xml_mode(path):
    """'lines' if each line holds one complete record, otherwise 'stream'. Checked once per file/archive."""
    if XML_MODE in ("lines", "stream"):
        return XML_MODE
    key = str(path)
    if key not in _XML_MODE_CACHE:
        _XML_MODE_CACHE[key] = _detect_xml_mode(path)
    return _XML_MODE_CACHE[key]

def _detect_xml_mode(path):
    head = peek_bytes(path, 2_000_000).decode("utf-8", errors="replace")
    lines = [l.strip() for l in head.splitlines() if l.strip() and not l.strip().startswith("<?xml")]
    rec_line = re.compile(r"^<(\w+:)?REC[\s>].*</(\w+:)?REC>$")
    if len(lines) >= 2 and rec_line.match(lines[0]) and rec_line.match(lines[1]):
        return "lines"
    return "stream"

if not WOS_FILES_DF.empty:
    SAMPLE_FILE = Path(WOS_FILES_DF.loc[WOS_FILES_DF.size_bytes.idxmin() if len(WOS_FILES_DF) > 50 else 0, "path"])
    print("Sample file:", SAMPLE_FILE, "|", human_size(SAMPLE_FILE.stat().st_size))
    print("Detected format:", data_format_for(SAMPLE_FILE.name),
          "| XML layout:", detect_xml_mode(SAMPLE_FILE) if data_format_for(SAMPLE_FILE.name) == "xml" else "n/a")
    text = peek_bytes(SAMPLE_FILE, 3000).decode("utf-8", errors="replace")
    print("\n--- first ~3,000 characters (long lines cut at 250 chars) ---")
    for line in text.splitlines()[:40]:
        print(line[:250] + (" …" if len(line) > 250 else ""))

### 2c. Run status — what is new, done, or still missing
This notebook is a **resumable pipeline**: WoS ZIP archives are *temporary inputs*; the small files in `OUTPUT_DIR`
are the *persistent state*:

| Persistent file | Role |
|---|---|
| `GREI_WoS_processing_log.csv` | which archives (by content fingerprint) were completed, when, with what counts |
| `GREI_WoS_citation_edges_master.csv` | every citation relationship found so far (accumulates across runs) |
| `GREI_WoS_archive_manifest.csv` | what each archive contained (dates, records, UTs, dataset), kept after the ZIP is deleted |
| `GREI-DataCite-NIHOnly_with_WoS_citations.xlsx` | your workbook + WoS metrics, rebuilt each run from the whole master |
| `archive_inventory/` | small per-archive date profiles / hashed UT lists (for coverage and overlap reports) |

A previously processed archive that is no longer in the workspace is **not an error**. Archives are only ever skipped
or processed — never deleted by the notebook.

In [ ]:
def build_expected_df():
    rows = []
    for item in EXPECTED_DATASETS:
        name, start, end = item[0], item[1], item[2]
        exp = item[3] if len(item) > 3 else None
        s_, e_ = pd.to_datetime(start, errors="coerce"), pd.to_datetime(end, errors="coerce")
        if pd.isna(s_) or pd.isna(e_) or s_ > e_:
            warn(f"EXPECTED_DATASETS entry {item} has an invalid date range and is ignored.")
            continue
        rows.append({"dataset_name": str(name).strip(), "start": s_.date(), "end": e_.date(),
                     "expected_records": int(exp) if exp else None})
    return pd.DataFrame(rows, columns=["dataset_name", "start", "end", "expected_records"]) \
        .sort_values("start").reset_index(drop=True)

EXPECTED_DF = build_expected_df()

def completed_status():
    # -> (completed content fingerprints, {file_key: size} for completed rows logged without a fingerprint).
    # A file/archive counts as done when its most recent log row is "completed".
    log = read_log()
    if log.empty:
        return set(), {}
    log = log.reset_index().sort_values("index")
    log["_id"] = [fp if fp else "KEY:" + k for fp, k in zip(log.content_fingerprint, log.file_key)]
    last = log.groupby("_id").tail(1)
    done = last[last.status == "completed"]
    fps = set(done.content_fingerprint[done.content_fingerprint != ""])
    legacy = {k: int(s) for k, s, fp in zip(done.file_key, done.file_size_bytes, done.content_fingerprint) if not fp}
    return fps, legacy

def is_completed(row, fps, legacy):
    return row.content_fingerprint in fps or legacy.get(row.file_key) == int(row.size_bytes)

def count_completed():
    fps, legacy = completed_status()
    return len(fps) + len(legacy)


def _latest_log_by_fp():
    log = read_log()
    log = log[log.content_fingerprint != ""].reset_index().sort_values("index")
    return log.groupby("content_fingerprint").tail(1).set_index("content_fingerprint") if len(log) else log

def _manifest_rows():
    if not MANIFEST_CSV.exists():
        return {}
    m = pd.read_csv(MANIFEST_CSV, dtype=str, keep_default_na=False)
    return {r["content_fingerprint"]: r for r in m.to_dict("records")}

def expected_status_table():
    fps_done, _ = completed_status()
    man = _manifest_rows()
    present = set(WOS_FILES_DF.content_fingerprint)
    rows = []
    for e in EXPECTED_DF.itertuples():
        done = [r for fp, r in man.items() if r.get("dataset_name") == e.dataset_name and fp in fps_done]
        cand = [r for fp, r in man.items() if fp in present and fp not in fps_done
                and e.dataset_name in (r.get("dataset_name"), r.get("suggested_dataset"))]
        if done:
            status = "processed" + ("" if all(r.get("dataset_assignment_source", "").startswith("manual") for r in done)
                                    else " (identified from XML dates - please confirm)")
        elif cand:
            status = "NOT yet processed - candidate archive present: " + ", ".join(r["current_file_name"] for r in cand)
        else:
            status = "NOT yet processed - archive not present"
        rows.append({"dataset_name": e.dataset_name, "start": e.start, "end": e.end, "status": status,
                     "archive(s)": MULTI_SEP.join(r.get("current_file_name", "") for r in done)})
    return pd.DataFrame(rows, columns=["dataset_name", "start", "end", "status", "archive(s)"])

def state_integrity_check():
    # Every completed archive that produced matches must be represented in the master (or the raw journal).
    P = output_paths(False)
    log = read_log()
    comp = log[(log.status == "completed") & (pd.to_numeric(log.unique_relationships, errors="coerce").fillna(0) > 0)]
    if comp.empty:
        print("No completed archives with citation matches yet - nothing to check.")
        return True
    have = set()
    for f, cols in [(P["master_csv"], ["all_source_archive_fingerprints", "source_archive_fingerprint"]),
                    (P["edges_raw"], ["source_archive_fingerprint"])]:
        if f.exists():
            hdr = _csv_header(f)
            use = [c for c in cols if c in hdr]
            if use:
                d = pd.read_csv(f, dtype=str, keep_default_na=False, usecols=use)
                for c in use:
                    for v in d[c]:
                        have |= _split_multi_set(v) if MULTI_SEP in v else ({v} if v else set())
            elif "source_wos_file" in hdr:
                have.add("__legacy__")
    missing = comp[(comp.content_fingerprint != "") & ~comp.content_fingerprint.isin(have)]
    missing = missing.drop_duplicates("content_fingerprint")
    if not P["master_csv"].exists() and not P["edges_raw"].exists():
        warn("The processing log lists completed archives with matches, but NO master or raw edge file exists. "
             "Restore GREI_WoS_citation_edges_master.csv (see state_backups/) before continuing.")
        return False
    if len(missing):
        warn(f"{len(missing)} completed archive(s) have no rows in the master. Restore the master from "
             "state_backups/, or download those exports again and list them in FORCE_REPROCESS_FILES:")
        display(missing[["file_key", "content_fingerprint", "finished_at", "unique_relationships"]])
        return False
    print("✅ Persistent state is consistent: every completed archive's matches are in the master.")
    return True

def _sha256_file(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

# ---- tables ------------------------------------------------------------------------------
fps_done, legacy_done = completed_status()
last_by_fp = _latest_log_by_fp()
man_rows = _manifest_rows()
rows, seen = [], {}
for r in WOS_FILES_DF.itertuples():
    fp = r.content_fingerprint
    last = last_by_fp.loc[fp] if len(last_by_fp) and fp in last_by_fp.index else None
    if fp in seen:
        status = f"duplicate of {seen[fp]} - will be skipped"
    elif is_completed(r, fps_done, legacy_done):
        when = last.finished_at if last is not None else ""
        status = f"previously processed ({when}) - safe to remove"
    elif last is not None and last.status in ("failed", "interrupted", "started"):
        status = f"previous attempt {last.status} - will be retried"
    else:
        status = "NEW - ready to process"
    seen.setdefault(fp, r.file_name)
    m = man_rows.get(fp, {})
    rows.append({"ZIP archive": r.file_key, "size": r.size, "fingerprint": r.fingerprint_short,
                 "dataset (if known)": m.get("dataset_name", ""), "status": status})
START_PRESENT = pd.DataFrame(rows, columns=["ZIP archive", "size", "fingerprint", "dataset (if known)", "status"])

present_fps = set(WOS_FILES_DF.content_fingerprint)
gone = []
for fp in sorted(fps_done - present_fps):
    m, last = man_rows.get(fp, {}), (last_by_fp.loc[fp] if fp in last_by_fp.index else None)
    gone.append({"archive (last known name)": m.get("current_file_name") or (last.file_key if last is not None else ""),
                 "fingerprint": fp[:12], "dataset": m.get("dataset_name", ""),
                 "completed": last.finished_at if last is not None else "",
                 "records": (last.xml_members or last.records_examined) if last is not None else "",
                 "relationships found": last.unique_relationships if last is not None else ""})
START_GONE = pd.DataFrame(gone)
log_all = read_log()
failed_gone = log_all[(log_all.content_fingerprint != "") & ~log_all.content_fingerprint.isin(present_fps | fps_done)]
failed_gone = failed_gone.reset_index().sort_values("index").groupby("content_fingerprint").tail(1) if len(failed_gone) else failed_gone

print("=" * 78)
print(f"RUN STATUS AT START  ({now_iso()})")
print("=" * 78)
print(f"\nZIP archives currently present: {len(START_PRESENT)}")
if len(START_PRESENT):
    display(START_PRESENT)
print(f"\n.part / incomplete downloads in progress: {len(INCOMPLETE_FILES)}")
for p_ in INCOMPLETE_FILES:
    print(f"   ⏳ {p_.name} ({human_size(_path_size(p_))}) - incomplete transfer, will not be processed")
print(f"\nPreviously processed archives no longer present (results kept): {len(START_GONE)}")
if len(START_GONE):
    display(START_GONE)
if len(failed_gone):
    print(f"\nArchives whose last attempt did NOT complete and that are not present now: {len(failed_gone)}")
    display(failed_gone[["file_key", "status", "started_at", "error"]])
if len(EXPECTED_DF):
    START_EXPECTED = expected_status_table()
    pend = START_EXPECTED[~START_EXPECTED.status.str.startswith("processed")]
    print(f"\nExpected date ranges still not processed: {len(pend)} of {len(START_EXPECTED)}")
    display(START_EXPECTED)
print(f"\nArchives completed in all runs so far: {count_completed():,}")
STATE_OK = state_integrity_check()

def restore_latest_backup():
    """Copy the newest state backup that contains a master back into OUTPUT_DIR. Run manually:  restore_latest_backup()"""
    root = OUTPUT_DIR / "state_backups"
    cands = [d for d in sorted(root.iterdir(), reverse=True)
             if (d / output_paths(False)["master_csv"].name).exists()] if root.exists() else []
    if not cands:
        print("No backup containing a master file was found.")
        return None
    for f in cands[0].iterdir():
        shutil.copy2(f, OUTPUT_DIR / f.name)
        print(f"restored {f.name} from {cands[0].name}")
    print("Now re-run the notebook from the top.")
    return cands[0]

if not STATE_OK:
    print("   -> To restore the most recent good copy, run  restore_latest_backup()  in a new cell, then re-run.")

if COMPUTE_FULL_FILE_SHA256:
    FULL_SHA = {}
    for r in WOS_FILES_DF.drop_duplicates("content_fingerprint").itertuples():
        if not man_rows.get(r.content_fingerprint, {}).get("full_file_sha256"):
            t0 = time.time()
            FULL_SHA[r.content_fingerprint] = _sha256_file(r.path)
            print(f"SHA-256 {r.file_name}: {FULL_SHA[r.content_fingerprint][:16]}… ({time.time() - t0:,.0f}s)")

### 2d. Read the target DOIs (`GREI-DataCite-NIHOnly.csv`)
Every column is read as **text**, so DOIs, counts and the JSON-style DataCite fields are kept exactly as delivered
(nothing is converted to numbers or dates). Comma- or tab-separated files are detected automatically, as is the text
encoding. Duplicate rows are **kept**. A `source_excel_row` column (the row number when the file is opened in Excel;
row 1 is the header) is added so every result can be traced back to its row.

In [ ]:
def load_target_table(path):
    """Read the target DOI file (CSV/TSV/TXT or Excel) with every column as text."""
    path = Path(path)
    if path.suffix.lower() in (".xlsx", ".xlsm", ".xls"):
        return pd.read_excel(path, sheet_name=TARGET_SHEET, dtype=str), "Excel"
    raw = path.read_bytes()[:200_000]
    for enc in ("utf-8-sig", "cp1252", "latin-1"):          # latin-1 never fails (last resort)
        try:
            head = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    first_line = head.splitlines()[0] if head else ""
    sep = "\t" if first_line.count("\t") > first_line.count(",") else ","
    df = pd.read_csv(path, sep=sep, dtype=str, encoding=enc, keep_default_na=False, na_values=[""])
    if enc != "utf-8-sig":
        warn(f"{path.name} is not UTF-8; read as {enc}. Accented characters in titles may look odd (DOIs unaffected).")
    return df, f"{'tab' if sep == chr(9) else 'comma'}-separated, {enc.replace('-sig', '')}"

t0 = time.time()
targets_df, TARGET_FORMAT = load_target_table(TARGET_PATH)
targets_df.columns = [str(c).strip() for c in targets_df.columns]
ORIGINAL_COLUMNS = list(targets_df.columns)
targets_df.insert(0, "source_excel_row", range(2, len(targets_df) + 2))
print(f"Loaded {TARGET_PATH.name} ({TARGET_FORMAT}) in {time.time() - t0:.1f}s")

if TARGET_DOI_COLUMN not in targets_df.columns:
    raise KeyError(f"Column {TARGET_DOI_COLUMN!r} not found. Columns are: {ORIGINAL_COLUMNS}")
HAVE_QA_COL = QA_DOI_COLUMN in targets_df.columns
if not HAVE_QA_COL:
    warn(f"QA column {QA_DOI_COLUMN!r} not found; the id-vs-attributes_doi check will be skipped.")

print("Shape (rows, columns):", targets_df.shape[0], ",", len(ORIGINAL_COLUMNS))
if SHOW_DETAILS:
    print("\nColumns:")
    for i, c in enumerate(ORIGINAL_COLUMNS, 1):
        print(f"  {i:>2}. {c}")

ids = targets_df[TARGET_DOI_COLUMN]
n_nonnull = int(ids.notna().sum())
n_unique = int(ids.dropna().nunique())
print(f"\nRows                         : {len(targets_df):,}")
print(f"Non-null '{TARGET_DOI_COLUMN}' values       : {n_nonnull:,}")
print(f"Unique '{TARGET_DOI_COLUMN}' values         : {n_unique:,}")
print(f"Rows with a duplicated '{TARGET_DOI_COLUMN}': {int(ids.dropna().duplicated(keep=False).sum()):,} "
      f"(extra copies: {n_nonnull - n_unique:,}) — all rows are preserved")
show_cols = [c for c in ["source_excel_row", TARGET_DOI_COLUMN, QA_DOI_COLUMN, "attributes_titles",
                         "attributes_publicationYear", "attributes_types_resourceTypeGeneral",
                         "relationships_client_data_id"] if c in targets_df.columns]
display(targets_df[show_cols].head(8))

### 2e. Clean up the target DOIs
One function, `normalize_doi`, is used for **both** the workbook DOIs and every DOI found in WoS cited references, so the
two sides are always cleaned identically. Matching is **exact** after normalization — no fuzzy matching.

Normalization: text → trim → lowercase → remove `https://doi.org/`, `http://dx.doi.org/`, `doi:`, `DOI ` etc. →
strip surrounding quotes/brackets and trailing citation punctuation (`. , ; :` and an unbalanced `)`) → must then look like
`10.<4–9 digits>/<suffix>` with no spaces, otherwise it is reported as unusable.

In [ ]:
_ZERO_WIDTH = dict.fromkeys(map(ord, "​‌‍⁠﻿"), None)
_DOI_PREFIXES = ("https://doi.org/", "http://doi.org/", "https://dx.doi.org/", "http://dx.doi.org/",
                 "https://www.doi.org/", "http://www.doi.org/", "doi.org/", "dx.doi.org/",
                 "urn:doi:", "doi:", "doi ", "doi/")
_LEAD_CHARS  = " \t\r\n\"'<[({“‘"
_TRAIL_CHARS = " \t\r\n.,;:\"'>]}”’"
_DOI_FULL_RE = re.compile(r"10\.\d{4,9}/\S+")
DOI_IN_TEXT_RE = re.compile(r"10\.\d{4,9}/[^\s\"<>]+", re.IGNORECASE)

def normalize_doi_with_reason(value):
    """Return (normalized_doi_or_None, reason). reason is 'ok', 'blank', 'not_a_doi' or 'contains_whitespace'."""
    if value is None:
        return None, "blank"
    if isinstance(value, float) and math.isnan(value):
        return None, "blank"
    s = str(value).translate(_ZERO_WIDTH).replace(" ", " ").strip().lower()
    if s in ("", "nan", "none", "null", "<na>", "n/a"):
        return None, "blank"
    for _ in range(6):                      # repeat until nothing more changes
        before = s
        s = s.strip().lstrip(_LEAD_CHARS)
        for p in _DOI_PREFIXES:
            if s.startswith(p):
                s = s[len(p):]
                break
        s = s.strip().rstrip(_TRAIL_CHARS)
        if s.endswith(")") and s.count("(") < s.count(")"):
            s = s[:-1]
        if s == before:
            break
    if not s:
        return None, "blank"
    if not s.startswith("10."):
        return None, "not_a_doi"
    if re.search(r"\s", s):
        return None, "contains_whitespace"
    if not _DOI_FULL_RE.fullmatch(s):
        return None, "not_a_doi"
    return s, "ok"

def normalize_doi(value):
    return normalize_doi_with_reason(value)[0]

# ---- quick self-test of the normalizer -------------------------------------------------
_tests = {
    "10.5061/DRYAD.ABC123": "10.5061/dryad.abc123",
    " https://doi.org/10.5061/dryad.abc123 ": "10.5061/dryad.abc123",
    "http://dx.doi.org/10.5061/dryad.abc123.": "10.5061/dryad.abc123",
    "doi:10.6084/m9.figshare.123.v2": "10.6084/m9.figshare.123.v2",
    "DOI 10.7910/DVN/XYZ,": "10.7910/dvn/xyz",
    "(10.1002/(SICI)1097-0258(19980815)17:15<1661::AID-SIM968>3.0.CO;2-2)":
        "10.1002/(sici)1097-0258(19980815)17:15<1661::aid-sim968>3.0.co;2-2",
    "[https://doi.org/10.18112/openneuro.ds000001.v1.0.0];": "10.18112/openneuro.ds000001.v1.0.0",
    "not a doi": None, "": None, None: None, float("nan"): None, "10.1234/has space": None,
}
for raw, expected in _tests.items():
    got = normalize_doi(raw)
    assert got == expected, f"normalize_doi({raw!r}) -> {got!r}, expected {expected!r}"
print(f"✅ normalize_doi passed {len(_tests)} self-tests")

In [ ]:
res = targets_df[TARGET_DOI_COLUMN].map(normalize_doi_with_reason)
targets_df["target_doi_normalized"] = res.map(lambda r: r[0])
targets_df["target_doi_normalization_status"] = res.map(lambda r: r[1])

TARGET_DOI_SET = set(targets_df["target_doi_normalized"].dropna())
usable = targets_df["target_doi_normalized"].notna()

print(f"Total rows                     : {len(targets_df):,}")
print(f"Rows with a usable target DOI  : {int(usable.sum()):,}")
print(f"Unique normalized target DOIs  : {len(TARGET_DOI_SET):,}")
dup_mask = targets_df["target_doi_normalized"].notna() & targets_df["target_doi_normalized"].duplicated(keep=False)
print(f"Rows sharing a normalized DOI  : {int(dup_mask.sum()):,} "
      f"({targets_df.loc[dup_mask, 'target_doi_normalized'].nunique():,} DOIs appear more than once)")
bad = targets_df[~usable]
print(f"Malformed / unusable '{TARGET_DOI_COLUMN}' : {len(bad):,}")
if len(bad):
    print("  by reason:", dict(Counter(bad["target_doi_normalization_status"])))
    display(bad[["source_excel_row", TARGET_DOI_COLUMN, "target_doi_normalization_status"]].head(15))

# DOIs that only become identical after normalization (case/prefix differences in the raw id)
raw_vs_norm = targets_df[usable].groupby("target_doi_normalized")[TARGET_DOI_COLUMN].nunique()
if (raw_vs_norm > 1).any():
    warn(f"{int((raw_vs_norm > 1).sum())} normalized DOIs come from differently-written raw ids (e.g. case differences).")

# ---- QA: id vs attributes_doi ------------------------------------------------------------
if HAVE_QA_COL:
    targets_df["attributes_doi_normalized"] = targets_df[QA_DOI_COLUMN].map(normalize_doi)
    both = targets_df["target_doi_normalized"].notna() & targets_df["attributes_doi_normalized"].notna()
    agree = both & (targets_df["target_doi_normalized"] == targets_df["attributes_doi_normalized"])
    differ = both & ~agree
    targets_df["id_vs_attributes_doi"] = "missing_one_or_both"
    targets_df.loc[agree, "id_vs_attributes_doi"] = "agree"
    targets_df.loc[differ, "id_vs_attributes_doi"] = "DIFFER"
    print(f"\nQA '{TARGET_DOI_COLUMN}' vs '{QA_DOI_COLUMN}' (after normalization; id remains authoritative):")
    print(f"  rows with both usable : {int(both.sum()):,}")
    print(f"  agree                 : {int(agree.sum()):,}")
    print(f"  differ                : {int(differ.sum()):,}")
    if differ.any():
        display(targets_df.loc[differ, ["source_excel_row", TARGET_DOI_COLUMN, QA_DOI_COLUMN]].head(15))

# helpers for the "possible version near-miss" report (never used for counting)
TARGET_PREFIXES = {d.split("/", 1)[0] for d in TARGET_DOI_SET}
VERSION_SUFFIX_RE = re.compile(r"^(.+?)((?:[./]v\d+(?:\.\d+)*)|(?:\.\d{1,2}))$")
TARGET_BASE_TO_TARGETS = defaultdict(set)
for d in TARGET_DOI_SET:
    m = VERSION_SUFFIX_RE.match(d)
    if m:
        TARGET_BASE_TO_TARGETS[m.group(1)].add(d)
print(f"\nTarget DOI registrant prefixes: {len(TARGET_PREFIXES):,} (e.g. {sorted(TARGET_PREFIXES)[:8]})")

### 2f. Check the format of the WoS records

#### Readers
These functions read one record at a time and free memory as they go:

* **XML (one big document)** – `iterparse`; each finished record is handed over, then cleared and detached from its parent.
* **XML (one record per line)** – each line is parsed on its own.
* **JSONL** – one JSON record per line; converted to the same element structure so one extractor works for both.
* **.gz** and **.zip** archives are decompressed on the fly; source files are only ever opened read-only.

In [ ]:
def local(tag):
    """Tag name without namespace: '{http://...}REC' -> 'REC'."""
    return tag.rsplit("}", 1)[-1] if isinstance(tag, str) else ""

def iter_streams(path, member_filter=None):
    """Yield (member_name, binary_stream, data_format). A ZIP yields one stream per XML member
    (ZIP_XML_MEMBER_SUFFIX); other entries are skipped. member_filter(index, n_members) can select a subset."""
    path = Path(path)
    if path.name.lower().endswith(".zip"):
        with zipfile.ZipFile(path) as zf:
            members = [i for i in zf.infolist() if is_zip_record_member(i)]
            for idx, info in enumerate(members):
                if member_filter is not None and not member_filter(idx, len(members)):
                    continue
                with zf.open(info) as raw:
                    yield info.filename, raw, "xml"
    else:
        opener = gzip.open if path.name.lower().endswith(".gz") else open
        with opener(path, "rb") as f:
            yield "", f, data_format_for(path.name)

def iter_xml_stream(stream, record_tag):
    if HAVE_LXML:
        ctx = LET.iterparse(stream, events=("end",), tag="{*}" + record_tag, huge_tree=True,
                            remove_comments=True, resolve_entities=False, no_network=True)
        for _, elem in ctx:
            yield elem
            elem.clear(keep_tail=True)
            while elem.getprevious() is not None:          # drop already-processed siblings
                del elem.getparent()[0]
        del ctx
    else:
        stack = []
        for event, elem in ET.iterparse(stream, events=("start", "end")):
            if event == "start":
                stack.append(elem)
                continue
            stack.pop()
            if local(elem.tag) == record_tag:
                yield elem
                elem.clear()
                if stack:
                    stack[-1].remove(elem)

def _parse_xml_text(data):
    return LET.fromstring(data, parser=LET.XMLParser(huge_tree=True, resolve_entities=False, no_network=True)) \
        if HAVE_LXML else ET.fromstring(data)

def iter_xml_lines(stream, record_tag, issues):
    for i, line in enumerate(stream, 1):
        line = line.strip()
        if not line or line.startswith(b"<?xml"):
            continue
        try:
            el = _parse_xml_text(line)
        except Exception as e:
            issues.append(f"line {i}: XML parse error {str(e)[:120]}")
            continue
        if local(el.tag) == record_tag:
            yield el
        else:
            for sub in el.iter():
                if local(sub.tag) == record_tag:
                    yield sub

def dict_to_element(tag, obj):
    """Convert a JSON record into an element tree. Scalars become attributes AND child elements;
    'content' / '#text' become element text. (Matches the Clarivate WoS JSON layout.)"""
    el = ET.Element(str(tag))
    if isinstance(obj, dict):
        has_text = any(str(k).lstrip("@") in ("content", "#text", "_text") for k in obj)
        for k, v in obj.items():
            k = str(k).lstrip("@")
            if k in ("content", "#text", "_text"):
                el.text = "" if v is None else str(v)
            elif isinstance(v, (dict, list)):
                for item in (v if isinstance(v, list) else [v]):
                    el.append(dict_to_element(k, item))
            else:
                if v is not None:
                    el.set(k, str(v))
                    if not has_text:              # keep text-bearing elements' text clean
                        ET.SubElement(el, k).text = str(v)
    elif obj is not None:
        el.text = str(obj)
    return el

def _find_record_dicts(obj, record_tag):
    if isinstance(obj, list):
        for it in obj:
            yield from _find_record_dicts(it, record_tag)
    elif isinstance(obj, dict):
        if record_tag in obj:
            v = obj[record_tag]
            for it in (v if isinstance(v, list) else [v]):
                yield it
        elif "UID" in obj or "static_data" in obj:
            yield obj
        else:
            for v in obj.values():
                if isinstance(v, (dict, list)):
                    yield from _find_record_dicts(v, record_tag)

def iter_json_stream(stream, fmt, record_tag, issues, size_bytes):
    if fmt == "jsonl":
        for i, line in enumerate(stream, 1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception as e:
                issues.append(f"line {i}: JSON error {str(e)[:120]}")
                continue
            for rec in _find_record_dicts(obj, record_tag):
                yield dict_to_element(record_tag, rec)
    else:
        if size_bytes > MAX_JSON_FILE_MB * 1024**2:
            raise MemoryError(f"Single-document JSON larger than MAX_JSON_FILE_MB={MAX_JSON_FILE_MB}; "
                              "it cannot be streamed safely. Convert to JSONL or raise the limit.")
        obj = json.load(stream)
        for rec in _find_record_dicts(obj, record_tag):
            yield dict_to_element(record_tag, rec)

MAX_INMEMORY_MEMBER = 64 << 20        # ZIP members up to 64 MB are parsed whole (a WoS record is ~10-100 KB)

def _child_path(el, path):
    for part in path.split("/"):
        el = next((c for c in el if local(c.tag) == part), None)
        if el is None:
            return None
    return el

def unwrap_proquest(el, issues):
    """ProQuest <RECORD> -> the embedded Clarivate <REC> (parsed from text). Other elements pass through."""
    if local(el.tag) != PROQUEST_WRAPPER_TAG:
        return el
    goid = (_child_path(el, "GOID").text or "").strip() if _child_path(el, "GOID") is not None else ""
    emb = _child_path(el, EMBEDDED_WOS_XML_PATH)
    txt = (emb.text or "").strip() if emb is not None else ""
    if "<" not in txt:
        issues.append(f"GOID {goid}: no embedded WoS XML at {EMBEDDED_WOS_XML_PATH}")
        return None
    try:
        rec = _parse_xml_text(txt.encode("utf-8"))
    except Exception as e:
        issues.append(f"GOID {goid}: embedded WoS XML could not be parsed ({str(e)[:100]})")
        return None
    if goid:
        rec.set("proquest_goid", goid)
    return rec

class _ChainReader:
    """File-like object: already-read bytes followed by the rest of a stream."""
    def __init__(self, head, rest):
        self.head, self.pos, self.rest = head, 0, rest
    def read(self, n=-1):
        if self.pos < len(self.head):
            if n is None or n < 0:
                out = self.head[self.pos:] + self.rest.read()
                self.pos = len(self.head)
                return out
            out = self.head[self.pos:self.pos + n]
            self.pos += len(out)
            return out
        return self.rest.read() if n is None or n < 0 else self.rest.read(n)

_RE_UID = re.compile(rb"(?:&lt;|<)UID(?:&gt;|>)\s*([^<&\s]+)")
_RE_ATTR = re.compile(rb'([A-Za-z_]+)=(?:"|&quot;)([^"&<>]*)(?:"|&quot;)')

def fast_record(data):
    """A minimal stand-in record (UID + pub_info dates) read from raw bytes without parsing the XML."""
    rec = ET.Element("REC", {"prefiltered": "1"})
    m = _RE_UID.search(data)
    if m:
        ET.SubElement(rec, "UID").text = m.group(1).decode("utf-8", "replace")
    i = data.find(b"pub_info")
    if i >= 0:
        seg = data[i:i + 800]
        cut = min([x for x in (seg.find(b"&gt;"), seg.find(b">")) if x >= 0] or [len(seg)])
        attrs = {k.decode(): v.decode("utf-8", "replace") for k, v in _RE_ATTR.findall(seg[:cut])}
        summ = ET.SubElement(ET.SubElement(rec, "static_data"), "summary")
        ET.SubElement(summ, "pub_info", attrs)
    return rec

def iter_zip_member(stream, record_tag, issues, prefilter=None):
    """One ZIP member: parsed whole when small (TDM Studio: one record per member), streamed when large.
    ProQuest <RECORD> wrappers are unwrapped to the embedded WoS <REC>. With `prefilter` (list of byte strings),
    a one-record ProQuest member containing none of them is returned as a lightweight stand-in (not parsed)."""
    data = stream.read(MAX_INMEMORY_MEMBER + 1)
    if (prefilter is not None and len(data) <= MAX_INMEMORY_MEMBER
            and f"<{PROQUEST_WRAPPER_TAG}>".encode() in data[:300]
            and not any(pfx in data for pfx in prefilter)):
        yield fast_record(data)
        return
    if len(data) <= MAX_INMEMORY_MEMBER:
        try:
            root = _parse_xml_text(data)
        except Exception as e:
            issues.append(f"XML parse error: {str(e)[:120]}")
            return
        tag = local(root.tag)
        if tag == PROQUEST_WRAPPER_TAG:
            rec = unwrap_proquest(root, issues)
            if rec is not None:
                yield rec
        elif tag == record_tag:
            yield root
        else:
            for sub in list(root.iter()):
                t = local(sub.tag)
                if t == record_tag:
                    yield sub
                elif t == PROQUEST_WRAPPER_TAG:
                    rec = unwrap_proquest(sub, issues)
                    if rec is not None:
                        yield rec
        return
    chained = _ChainReader(data, stream)
    if f"<{PROQUEST_WRAPPER_TAG}".encode() in data[:20000]:
        for el in iter_xml_stream(chained, PROQUEST_WRAPPER_TAG):
            rec = unwrap_proquest(el, issues)
            if rec is not None:
                yield rec
    else:
        yield from iter_xml_stream(chained, record_tag)

def first_zip_record(path):
    """(outer root tag, unwrapped record) of the first XML member of a ZIP, or (None, None)."""
    try:
        with zipfile.ZipFile(path) as zf:
            info = next((i for i in zf.infolist() if is_zip_record_member(i)), None)
            if info is None:
                return None, None
            with zf.open(info) as f:
                root = _parse_xml_text(f.read(MAX_INMEMORY_MEMBER))
        return local(root.tag), unwrap_proquest(root, [])
    except Exception:
        return None, None

def records_from_stream(path, stream, fmt, record_tag, issues, size, prefilter=None):
    """Record generator for one stream (one file, or one member of a ZIP)."""
    if fmt == "xml" and str(path).lower().endswith(".zip"):
        return iter_zip_member(stream, record_tag, issues, prefilter)
    if fmt == "xml":
        mode = detect_xml_mode(path)              # for .zip this looks at the first member
        return iter_xml_lines(stream, record_tag, issues) if mode == "lines" else iter_xml_stream(stream, record_tag)
    return iter_json_stream(stream, fmt, record_tag, issues, size)

def iter_records(path, record_tag, issues=None, prefilter=None):
    """Yield (member_name, record_element) one at a time from any supported WoS file."""
    issues = [] if issues is None else issues
    path = Path(path)
    size = path.stat().st_size
    for member, stream, fmt in iter_streams(path):
        for rec in records_from_stream(path, stream, fmt, record_tag, issues, size, prefilter):
            yield member, rec

#### Structure of a sample of records
First, the outer structure of the file (to find the element that represents one record), then a profile of the paths
**inside** the sampled records: how many records contain each path, how often it occurs, its attributes and a short example
value. Nothing large is printed.

In [ ]:
def profile_xml_head(path, max_elements=150_000):
    """Count element paths at the top of an XML file (stops early; does not read the whole file)."""
    counts = Counter()
    stack = []
    for member, stream, fmt in iter_streams(path):
        if fmt != "xml":
            return counts
        if detect_xml_mode(path) == "lines":
            for line in stream:
                line = line.strip()
                if line and not line.startswith(b"<?xml"):
                    counts["(one record per line) " + local(_parse_xml_text(line).tag)] += 1
                    if sum(counts.values()) > 50:
                        break
            return counts
        parser = (LET.iterparse(stream, events=("start", "end"), huge_tree=True) if HAVE_LXML
                  else ET.iterparse(stream, events=("start", "end")))
        n = 0
        for event, el in parser:
            if event == "start":
                stack.append(local(el.tag))
                if len(stack) <= 3:
                    counts["/".join(stack)] += 1
                n += 1
                if n >= max_elements:
                    break
            else:
                stack.pop()
                if len(stack) >= 2:
                    el.clear()
        break
    return counts

if not WOS_FILES_DF.empty and data_format_for(SAMPLE_FILE.name) == "xml":
    head_counts = profile_xml_head(SAMPLE_FILE)
    if SHOW_DETAILS:
        print("Outer structure (paths up to depth 3, counted over the first part of the file):")
        for path_, c in sorted(head_counts.items()):
            print(f"  {c:>8,}  {path_}")
    repeated = [p.split("/")[-1] for p, c in head_counts.items() if c > 1]
    RECORD_TAG_GUESS = "REC" if "REC" in repeated else (Counter(repeated).most_common(1)[0][0] if repeated else "REC")
    outer, inner = first_zip_record(SAMPLE_FILE) if SAMPLE_FILE.name.lower().endswith(".zip") else (None, None)
    if outer == PROQUEST_WRAPPER_TAG and inner is not None:
        RECORD_TAG_GUESS = local(inner.tag)
        print(f"\n✅ ProQuest TDM Studio format: each <{outer}> wraps the original Clarivate WoS record "
              f"<{RECORD_TAG_GUESS}> as text in {EMBEDDED_WOS_XML_PATH}. The notebook unwraps it, so the profile "
              "below shows the WoS record itself.")
        print("   Embedded record's top-level parts:", ", ".join(sorted({local(c.tag) for c in inner})))
    elif outer == PROQUEST_WRAPPER_TAG:
        warn(f"ProQuest <{outer}> records found, but no parseable WoS XML at {EMBEDDED_WOS_XML_PATH}. "
             "Check EMBEDDED_WOS_XML_PATH against the outer structure above.")
    elif outer:
        RECORD_TAG_GUESS = outer if outer == "REC" else RECORD_TAG_GUESS
    print("\nElement treated as ONE WoS record:", RECORD_TAG_GUESS)
else:
    RECORD_TAG_GUESS = "REC"

In [ ]:
def sample_records(path, record_tag, n):
    out, issues = [], []
    for member, rec in iter_records(path, record_tag, issues):
        out.append(copy.deepcopy(rec))          # keep a private copy; the stream clears the original
        if len(out) >= n:
            break
    return out, issues

def profile_records(recs, max_depth=8, top=400):
    doc_freq, occ, attrs, example = Counter(), Counter(), defaultdict(Counter), {}
    def walk(el, path, seen, depth):
        p = f"{path}/{local(el.tag)}" if path else local(el.tag)
        occ[p] += 1
        seen.add(p)
        for a, v in el.attrib.items():
            attrs[p][f"@{local(a)}={str(v)[:25]}" if local(a) in ("type", "role", "pref", "pubtype") else f"@{local(a)}"] += 1
        t = (el.text or "").strip()
        if t and p not in example:
            example[p] = t[:70]
        if depth < max_depth:
            for ch in el:
                if isinstance(ch.tag, str):
                    walk(ch, p, seen, depth + 1)
    for r in recs:
        seen = set()
        walk(r, "", seen, 0)
        doc_freq.update(seen)
    rows = [{"path": p, "records_with_path_%": round(100 * doc_freq[p] / max(len(recs), 1), 1),
             "occurrences": occ[p],
             "attributes": ", ".join(f"{k}" for k, _ in attrs[p].most_common(6)),
             "example_text": example.get(p, "")} for p in occ]
    return pd.DataFrame(rows).sort_values("path").head(top).reset_index(drop=True)

if not WOS_FILES_DF.empty:
    SAMPLE_RECS, sample_issues = sample_records(SAMPLE_FILE, RECORD_TAG_GUESS, INSPECT_SAMPLE_RECORDS)
    print(f"Sampled {len(SAMPLE_RECS)} records using record tag {RECORD_TAG_GUESS!r}. Parse issues: {len(sample_issues)}")
    for i in sample_issues[:5]:
        print("   ", i)
    if not SAMPLE_RECS:
        warn("No records were read. Look at the outer structure above and set SCHEMA['record_tag'] in the next cell.")
    else:
        PROFILE_DF = profile_records(SAMPLE_RECS)
        if SHOW_DETAILS:
            with pd.option_context("display.max_rows", 500, "display.max_colwidth", 70):
                display(PROFILE_DF)
        PROFILE_DF.to_csv(OUTPUT_DIR / "wos_schema_profile_sample.csv", index=False)
        print("Record structure saved to", OUTPUT_DIR / "wos_schema_profile_sample.csv",
              "(set SHOW_DETAILS = True to display it here)")

**TDM Studio exports** wrap each WoS record in ProQuest XML (`<RECORD>` with `<GOID>`, `<Obj>`, `<PublicationInfo>`).
The ProQuest part has no reference list; the **original Clarivate WoS record** (`<REC>`, with `refs/ref`) is stored as
text in `Obj/Notes/Note/RawFilename`. The notebook detects the wrapper, parses that embedded record, and works on it, so
the schema map below (standard Clarivate paths) applies. The ProQuest `GOID` is kept as `citing_proquest_goid`.

#### Where each field lives (schema map)
The defaults below follow the Clarivate Web of Science XML ("FullRecord", `wok5`) layout that TDM Studio delivers.
Paths are relative to one record and **ignore XML namespaces**. If the profile shows different names, edit them here and
re-run this cell and the validation cell below.

| Field | Default location in each record |
|---|---|
| WoS UT | `UID` |
| citing DOI | `dynamic_data/cluster_related/identifiers/identifier[@type='doi' or 'xref_doi']/@value` |
| year / date / type | `static_data/summary/pub_info/@pubyear, @sortdate, @pubtype` |
| title / source | `static_data/summary/titles/title[@type='item' / 'source']` |
| authors | `static_data/summary/names/name[@role='author']` |
| document type | `static_data/summary/doctypes/doctype` |
| affiliations | `static_data/fullrecord_metadata/addresses/address_name/address_spec` (full_address, organizations, city, state, country) |
| corresponding author | `names/name[@reprint='Y']` + `static_data/fullrecord_metadata/reprint_addresses/address_name` |
| WoS Categories / Research Areas | `static_data/fullrecord_metadata/category_info/subjects/subject[@ascatype='traditional' / 'extended']` |
| author keywords / Keywords Plus | `static_data/fullrecord_metadata/keywords/keyword` / `static_data/item/keywords_plus/keyword` |
| abstract | `static_data/fullrecord_metadata/abstracts/abstract/abstract_text` |
| ISSN / eISSN | `dynamic_data/cluster_related/identifiers/identifier[@type='issn' / 'eissn']` |
| funding | `static_data/fullrecord_metadata/fund_ack`: `fund_text` (raw acknowledgment) and `grants/grant` (`grant_agency` / `grant_agency_name`, `pref="Y"` = preferred name; anything tagged *parent*; `grant_ids/grant_id`) |
| **cited references** | `static_data/fullrecord_metadata/refs/ref` → DOI in child `doi` (and, optionally, anywhere in the reference text) |

Funding markup has changed across WoS schema versions, so check the `fund_ack` paths in the profile above. Every grant is
also stored verbatim as JSON (`citing_funding_grants_json`), so nothing is lost even if a sub-element is named differently.

In [ ]:
SCHEMA = {
    "record_tag": RECORD_TAG_GUESS,             # one WoS record (normally "REC")
    "ut_paths": ["UID"],                        # first non-empty wins

    "identifier_path": "dynamic_data/cluster_related/identifiers/identifier",
    "citing_doi_types": ["doi", "xref_doi"],    # in order of preference
    "identifier_type_attr": "type",
    "identifier_value_attr": "value",

    "pub_info_path": "static_data/summary/pub_info",
    "pubyear_attr": "pubyear", "sortdate_attr": "sortdate",
    "pubmonth_attr": "pubmonth", "pubtype_attr": "pubtype",
    "early_access_date_attr": "early_access_date",

    "title_path": "static_data/summary/titles/title",
    "title_type_attr": "type", "item_title_type": "item", "source_title_types": ["source"],

    "name_path": "static_data/summary/names/name",
    "name_role_attr": "role", "author_roles": ["author"],
    "author_name_tags": ["display_name", "full_name", "wos_standard"],   # first available is used
    "name_addr_no_attr": "addr_no",

    "doctype_path": "static_data/summary/doctypes/doctype",

    "address_path": "static_data/fullrecord_metadata/addresses/address_name/address_spec",
    "address_no_attr": "addr_no",
    "address_full_tag": "full_address",
    "organization_path": "organizations/organization",
    "organization_pref_attr": "pref",           # pref="Y" marks the preferred (unified) org name
    "city_tag": "city", "state_tag": "state", "country_tag": "country",

    # --- extra citing-publication metadata (only read for records that cite a target DOI) ---
    "reprint_attr": "reprint",                  # names/name[@reprint="Y"] = corresponding author
    "reprint_address_path": "static_data/fullrecord_metadata/reprint_addresses/address_name",
    "subject_path": "static_data/fullrecord_metadata/category_info/subjects/subject",
    "subject_type_attr": "ascatype",
    "wos_category_types": ["traditional"],      # Web of Science Categories
    "research_area_types": ["extended"],        # Research Areas
    "author_keyword_path": "static_data/fullrecord_metadata/keywords/keyword",
    "keywords_plus_path": "static_data/item/keywords_plus/keyword",
    "abstract_path": "static_data/fullrecord_metadata/abstracts/abstract/abstract_text",
    "issn_types": ["issn"], "eissn_types": ["eissn"],
    "fund_text_path": "static_data/fullrecord_metadata/fund_ack/fund_text",
    "grant_path": "static_data/fullrecord_metadata/fund_ack/grants/grant",
    "grant_agency_tags": ["grant_agency", "grant_agency_name"],   # pref="Y" marks the preferred/unified name
    "grant_id_path": "grant_ids/grant_id",

    # cited-reference list: newer WoS XML (your TDM Studio export) uses references/reference, older uses refs/ref.
    # Every path in the list is tried; the first that exists in a record is used.
    "refs_paths": ["static_data/fullrecord_metadata/references/reference",
                   "static_data/fullrecord_metadata/refs/ref"],
    "ref_doi_tags": ["doi"],                    # child elements of a reference that hold its DOI
    "ref_doi_attrs": ["doi"],                   # attributes of a reference that hold its DOI (if any)
    "ref_summary_tags": ["uid", "citedAuthor", "year", "citedTitle", "citedWork",
                         "volume", "page", "art_no", "doi"],
}
def get_refs(rec):
    for path in SCHEMA["refs_paths"]:
        refs = find_path(rec, path)
        if refs:
            return refs
    return []
SCHEMA["refs_path"] = SCHEMA["refs_paths"][0]      # used for messages / provenance only
print("Record tag:", SCHEMA["record_tag"], "| references looked for at:", " or ".join(SCHEMA["refs_paths"]))

In [ ]:
# ----- field extraction (namespace-agnostic, works for lxml and xml.etree) ---------------------
def children(el, name):
    return [c for c in el if local(c.tag) == name]

def find_path(el, path):
    cur = [el]
    for part in path.split("/"):
        nxt = []
        for e in cur:
            nxt.extend(children(e, part))
        cur = nxt
        if not cur:
            break
    return cur

def first(el, path):
    r = find_path(el, path)
    return r[0] if r else None

def text_of(el):
    if el is None:
        return ""
    return re.sub(r"\s+", " ", "".join(el.itertext())).strip()

def get_attr(el, name):
    if el is None:
        return ""
    for k, v in el.attrib.items():
        if local(k) == name:
            return (v or "").strip()
    return ""

def uniq(seq):
    seen, out = set(), []
    for x in seq:
        if x and x not in seen:
            seen.add(x)
            out.append(x)
    return out

def get_ut(rec):
    for p in SCHEMA["ut_paths"]:
        t = text_of(first(rec, p))
        if t:
            return t
    return ""

def extract_citing(rec):
    S = SCHEMA
    out = {"citing_wos_ut": get_ut(rec), "citing_proquest_goid": get_attr(rec, "proquest_goid")}

    found, issn, eissn = {}, [], []
    for idel in find_path(rec, S["identifier_path"]):
        t = get_attr(idel, S["identifier_type_attr"]).lower()
        val = get_attr(idel, S["identifier_value_attr"]) or text_of(idel)
        if t in S["citing_doi_types"] and t not in found:
            d = normalize_doi(val)
            if d:
                found[t] = d
        elif t in S["issn_types"]:
            issn.append(val)
        elif t in S["eissn_types"]:
            eissn.append(val)
    out["citing_doi"] = next((found[t] for t in S["citing_doi_types"] if t in found), "")
    out["citing_issn"] = MULTI_SEP.join(uniq(issn))
    out["citing_eissn"] = MULTI_SEP.join(uniq(eissn))

    pub = first(rec, S["pub_info_path"])
    year = get_attr(pub, S["pubyear_attr"])
    date = get_attr(pub, S["sortdate_attr"])
    if not year and date[:4].isdigit():
        year = date[:4]
    if not date:
        date = " ".join(x for x in [get_attr(pub, S["pubmonth_attr"]), year] if x)
    out["citing_year"] = year
    out["citing_publication_date"] = date
    out["citing_pubtype"] = get_attr(pub, S["pubtype_attr"])

    item, sources = "", []
    for t in find_path(rec, S["title_path"]):
        typ = get_attr(t, S["title_type_attr"]).lower()
        if typ == S["item_title_type"] and not item:
            item = text_of(t)
        elif typ in S["source_title_types"]:
            sources.append(text_of(t))
    out["citing_title"] = item
    out["citing_source"] = sources[0] if sources else ""

    authors, links, reprint_names = [], [], []
    for n in find_path(rec, S["name_path"]):
        if get_attr(n, S["name_role_attr"]).lower() not in S["author_roles"]:
            continue
        nm = person_name(n)
        authors.append(nm)
        if get_attr(n, S["reprint_attr"]).upper() == "Y":
            reprint_names.append(nm)
        a_no = get_attr(n, S["name_addr_no_attr"])
        links.append(f"{nm} [{','.join(a_no.split())}]" if a_no else nm)
    out["citing_authors"] = MULTI_SEP.join(authors)
    out["citing_author_address_links"] = MULTI_SEP.join(links)

    out["citing_document_type"] = MULTI_SEP.join(uniq(text_of(d) for d in find_path(rec, S["doctype_path"])))

    raw_addrs, orgs, orgs_orig, cities, regions, countries = [], [], [], [], [], []
    for a in find_path(rec, S["address_path"]):
        no = get_attr(a, S["address_no_attr"])
        full = text_of(first(a, S["address_full_tag"]))
        if full:
            raw_addrs.append(f"[{no}] {full}" if no else full)
        org_els = find_path(a, S["organization_path"])
        pref = [text_of(o) for o in org_els if get_attr(o, S["organization_pref_attr"]).upper() == "Y"]
        orgs.extend(pref if pref else [text_of(o) for o in org_els])
        orgs_orig.extend(text_of(o) for o in org_els if get_attr(o, S["organization_pref_attr"]).upper() != "Y")
        cities.append(text_of(first(a, S["city_tag"])))
        regions.append(text_of(first(a, S["state_tag"])))
        countries.append(text_of(first(a, S["country_tag"])))
    out["citing_affiliations_raw"] = ADDR_SEP.join(raw_addrs)
    out["citing_organizations"] = MULTI_SEP.join(uniq(orgs))
    out["citing_organizations_original"] = MULTI_SEP.join(uniq(orgs_orig))
    out["citing_cities"] = MULTI_SEP.join(uniq(cities))
    out["citing_regions"] = MULTI_SEP.join(uniq(regions))
    out["citing_countries"] = MULTI_SEP.join(uniq(countries))

    # corresponding (reprint) author: flagged names + reprint address block
    rp_addrs, rp_orgs, rp_countries = [], [], []
    for an in find_path(rec, S["reprint_address_path"]):
        reprint_names.extend(person_name(n) for n in find_path(an, "names/name"))
        for a in find_path(an, "address_spec"):
            full = text_of(first(a, S["address_full_tag"]))
            if full:
                rp_addrs.append(full)
            org_els = find_path(a, S["organization_path"])
            pref = [text_of(o) for o in org_els if get_attr(o, S["organization_pref_attr"]).upper() == "Y"]
            rp_orgs.extend(pref if pref else [text_of(o) for o in org_els])
            rp_countries.append(text_of(first(a, S["country_tag"])))
    out["citing_corresponding_authors"] = MULTI_SEP.join(uniq(reprint_names))
    out["citing_corresponding_address_raw"] = ADDR_SEP.join(uniq(rp_addrs))
    out["citing_corresponding_organizations"] = MULTI_SEP.join(uniq(rp_orgs))
    out["citing_corresponding_countries"] = MULTI_SEP.join(uniq(rp_countries))

    # subject classification and keywords (stored as given by WoS)
    subj = find_path(rec, S["subject_path"])
    stype = lambda x: get_attr(x, S["subject_type_attr"]).lower()
    out["citing_wos_categories"] = MULTI_SEP.join(uniq(text_of(x) for x in subj if stype(x) in S["wos_category_types"]))
    out["citing_research_areas"] = MULTI_SEP.join(uniq(text_of(x) for x in subj if stype(x) in S["research_area_types"]))
    out["citing_author_keywords"] = MULTI_SEP.join(uniq(text_of(x) for x in find_path(rec, S["author_keyword_path"])))
    out["citing_keywords_plus"] = MULTI_SEP.join(uniq(text_of(x) for x in find_path(rec, S["keywords_plus_path"])))
    out["citing_abstract"] = " ".join(t for t in (paragraphs_text(x) for x in find_path(rec, S["abstract_path"])) if t)

    # funding: original + preferred agency names, parent agency, grant numbers, raw acknowledgment text
    a_orig, a_pref, a_parent, gids, grants = [], [], [], [], []
    for g in find_path(rec, S["grant_path"]):
        g_orig, g_pref, g_par = [], [], []
        for d in g.iter():
            ln = local(d.tag)
            if ln in S["grant_agency_tags"]:
                t = text_of(d)
                (g_pref if get_attr(d, "pref").upper() == "Y" else g_orig).append(t)
                g_par.extend(v for k, v in d.attrib.items() if "parent" in local(k).lower() and v)
            elif "parent" in ln.lower():
                g_par.append(text_of(d))
        ids = [text_of(x) for x in find_path(g, S["grant_id_path"]) if text_of(x)]
        a_orig += g_orig; a_pref += g_pref; a_parent += g_par; gids += ids
        grants.append({"agency_original": uniq(g_orig), "agency_preferred": uniq(g_pref),
                       "agency_parent": uniq(g_par), "grant_ids": ids, "xml": element_to_compact(g)})
    out["citing_funding_agencies_original"] = MULTI_SEP.join(uniq(a_orig))
    out["citing_funding_agencies_preferred"] = MULTI_SEP.join(uniq(a_pref))
    out["citing_funding_agencies_parent"] = MULTI_SEP.join(uniq(a_parent))
    out["citing_grant_numbers"] = MULTI_SEP.join(uniq(gids))
    out["citing_funding_text"] = " ".join(t for t in (paragraphs_text(x) for x in find_path(rec, S["fund_text_path"])) if t)
    out["citing_funding_grants_json"] = json.dumps(grants, ensure_ascii=False) if grants else ""
    return out

def paragraphs_text(el):
    ps = children(el, "p")
    return " ".join(t for t in (text_of(p) for p in ps) if t) if ps else text_of(el)

def person_name(n):
    S = SCHEMA
    return next((text_of(first(n, t)) for t in S["author_name_tags"] if text_of(first(n, t))), "") or text_of(n)

def element_to_compact(el):
    """Small JSON-able copy of an XML element (attributes as @name, text as #text); keeps every WoS value."""
    d = {f"@{local(k)}": v for k, v in el.attrib.items()}
    kids = [c for c in el if isinstance(c.tag, str)]
    t = (el.text or "").strip()
    if not kids and not d:
        return t
    if t:
        d["#text"] = t
    for c in kids:
        d.setdefault(local(c.tag), []).append(element_to_compact(c))
    return d

def ref_summary(ref, limit=400):
    parts = []
    for c in ref:
        ln = local(c.tag)
        if ln in SCHEMA["ref_summary_tags"]:
            t = text_of(c)
            if t:
                parts.append(f"{ln}={t}")
    s = "; ".join(parts) or text_of(ref)
    return s[:limit]

def ref_dois(ref):
    """DOIs in one cited reference -> ({normalized: (method, raw_as_found)}, [malformed raw strings])."""
    found, malformed = {}, []
    for c in ref:
        if local(c.tag) in SCHEMA["ref_doi_tags"]:
            raw = text_of(c)
            if not raw:
                continue
            n = normalize_doi(raw)
            if n:
                found.setdefault(n, ("ref_doi_field", raw))
            else:
                hits = DOI_IN_TEXT_RE.findall(raw)       # e.g. two DOIs or junk around one DOI
                good = [normalize_doi(h) for h in hits]
                good = [g for g in good if g]
                for g in good:
                    found.setdefault(g, ("ref_doi_field", raw))
                if not good:
                    malformed.append(raw)
    for a in SCHEMA["ref_doi_attrs"]:
        raw = get_attr(ref, a)
        if raw:
            n = normalize_doi(raw)
            if n:
                found.setdefault(n, ("ref_doi_attribute", raw))
            else:
                malformed.append(raw)
    if SCAN_ALL_REFERENCE_TEXT_FOR_DOIS:
        txt = " ".join(list(ref.itertext()) + list(ref.attrib.values()))
        if "10." in txt:
            for m in DOI_IN_TEXT_RE.findall(txt):
                n = normalize_doi(m)
                if n:
                    found.setdefault(n, ("ref_text_scan", m))
    return found, malformed

#### Check: are references and fields found in the sample?
Coverage near 100% for UT, year, title and source is expected. **Cited references must be found** — if `records with ≥1
cited reference` is 0%, `SCHEMA["refs_paths"]` does not match your data and nothing will match.

In [ ]:
def validate_schema(recs):
    if not recs:
        warn("No sample records available.")
        return None
    meta = [extract_citing(r) for r in recs]
    n_refs = [len(get_refs(r)) for r in recs]
    ref_doi_counts, examples = [], []
    for r in recs:
        k = 0
        for ref in get_refs(r):
            f, _ = ref_dois(ref)
            if f:
                k += 1
                if len(examples) < 8:
                    examples.append({"citing_ut": get_ut(r), "cited_dois_found": ", ".join(f),
                                     "how": ", ".join(sorted({v[0] for v in f.values()})),
                                     "reference": ref_summary(ref, 160)})
        ref_doi_counts.append(k)
    cov = {f: round(100 * sum(1 for m in meta if m[f]) / len(meta), 1) for f in meta[0]}
    cov_df = pd.DataFrame({"field": list(cov), "records_with_value_%": list(cov.values())})
    print(f"Sample records: {len(recs)}")
    if SHOW_DETAILS:
        display(cov_df)
    else:
        weak = cov_df[cov_df["records_with_value_%"] < 50]
        print(f"Fields about the citing paper found in ≥50% of sample records: {len(cov_df) - len(weak)} of {len(cov_df)}"
              + (f" (less common: {', '.join(weak.field.str.replace('citing_', ''))})" if len(weak) else ""))
    total_refs = sum(n_refs)
    print(f"Records with ≥1 cited reference : {100 * sum(1 for x in n_refs if x) / len(recs):.1f}%")
    print(f"Cited references per record     : mean {total_refs / len(recs):.1f}, max {max(n_refs)}")
    print(f"Cited references with a DOI     : {sum(ref_doi_counts):,} of {total_refs:,} "
          f"({100 * sum(ref_doi_counts) / max(total_refs, 1):.1f}%)")
    if SHOW_DETAILS:
        print("\nOne extracted citing record (first sample, long values shortened):")
        for k, v in meta[0].items():
            print(f"  {k:<30}: {str(v)[:150]}")
    if examples:
        print("\nExample cited references that contain DOIs:")
        display(pd.DataFrame(examples).head(4 if not SHOW_DETAILS else 8))
    if total_refs == 0:
        print("\n❌ NO CITED REFERENCES FOUND. Add the reference-list path from the structure profile to "
              "SCHEMA['refs_paths'] before continuing.")
    elif sum(ref_doi_counts) == 0:
        warn("Cited references were found but none contain a DOI in this sample. Check SCHEMA['ref_doi_tags'].")
    else:
        print("\n✅ Cited references and cited-reference DOIs are being found.")
    return cov_df

if not WOS_FILES_DF.empty:
    SCHEMA_COVERAGE = validate_schema(SAMPLE_RECS)

### 2g. Which TDM Studio dataset is this export?
TDM Studio exports arrive with generic names (`wos-data.zip`, `wos-data (1).zip`, …), so the notebook identifies each archive
from its **contents**:

* **Content fingerprint** – computed from the ZIP's internal directory (member names, CRC-32 checksums, sizes). It does not
  change when a file is renamed, and an identical re-download gets the same fingerprint, so an archive is never processed twice
  by accident. A new download that reuses an old generic name gets a *different* fingerprint and is processed.
* **Publication-date profile** – read from each record's `pub_info`. Precision is recorded per record: `day` (e.g. `MAY 15`),
  `month` (`MAY`), `approx_month` (seasons/ranges such as `SPR`, `JAN-FEB`; WoS's sort-date month is used), or `year`.
* **Record count** – in TDM Studio ZIP exports each `.xml` member is one WoS record, so the number of XML members (read from
  the ZIP directory in seconds) is the **true record count**, matching the TDM Studio document count. Non-XML entries are
  counted and ignored. During processing the parsed record count is checked against it (`record_count_check`).
* **Inventory levels** – `sample` = a fast spread-out sample before processing (good for identification, provisional);
  `full` = every record, collected automatically during the citation run at no extra cost (exact record and UT counts).

**Provenance, most to least reliable:** (1) your manual assignment (`ARCHIVE_ASSIGNMENTS` or the `manual_*` columns of
`GREI_WoS_archive_manifest.csv`), (2) the current file name, (3) the XML publication-date distribution and record counts,
(4) UT overlap with other archives. Publication dates are **diagnostic**: they may not match the exact TDM Studio query
boundaries, so a suggestion is never treated as a confirmed assignment, and ambiguous cases are flagged rather than assigned.

In [ ]:
MONTH_NUM = {m: i for i, m in enumerate(["JAN", "FEB", "MAR", "APR", "MAY", "JUN",
                                         "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"], 1)}
PRECISION_ORDER = ["day", "month", "approx_month", "year", "none"]
_ISO_RE = re.compile(r"^(\d{4})-(\d{2})-(\d{2})$")

def parse_iso(s):
    m = _ISO_RE.match(str(s or "").strip())
    if not m:
        return None
    try:
        return date(int(m[1]), int(m[2]), int(m[3]))
    except ValueError:
        return None

def record_pub_date(pub):
    """(precision, value) for one record's pub_info element.
    WoS fills sortdate with a day even when only a month or year is known, so precision comes from pubmonth."""
    if pub is None:
        return "none", ""
    year = get_attr(pub, SCHEMA["pubyear_attr"])[:4]
    sd = get_attr(pub, SCHEMA["sortdate_attr"])
    sdd = parse_iso(sd)
    pm = re.sub(r"\s+", " ", get_attr(pub, SCHEMA["pubmonth_attr"]).upper()).strip()
    if not year.isdigit():
        year = str(sdd.year) if sdd else ""
    if not year:
        return "none", ""
    same_year = sdd is not None and str(sdd.year) == year
    if pm:
        m = re.match(r"^([A-Z]{3})[A-Z]*\.?(?: (\d{1,2}))?$", pm)
        if m and m.group(1) in MONTH_NUM:
            mo = MONTH_NUM[m.group(1)]
            if m.group(2):
                try:
                    return "day", date(int(year), mo, int(m.group(2))).isoformat()
                except ValueError:
                    pass
            return "month", f"{year}-{mo:02d}"
        return ("approx_month", sd[:7]) if same_year else ("year", year)
    if same_year:
        return ("year", year) if sd.endswith("-01-01") else ("day", sd)
    return "year", year

def bucket_interval(prec, val):
    """Earliest and latest calendar day a date bucket could mean."""
    if prec == "day":
        d = date.fromisoformat(val)
        return d, d
    if prec in ("month", "approx_month"):
        y, m = int(val[:4]), int(val[5:7])
        return date(y, m, 1), date(y + (m == 12), m % 12 + 1, 1) - timedelta(days=1)
    if prec == "year":
        y = int(val)
        return date(y, 1, 1), date(y, 12, 31)
    return None

def ut_hash(ut):
    return int.from_bytes(hashlib.blake2b(ut.encode("utf-8"), digest_size=8).digest(), "big")

class InventoryAccumulator:
    """What an archive contains, stored compactly: UTs as 8-byte hashes plus publication-date counts."""
    def __init__(self, level):
        self.level = level
        self.records = self.no_ut = self.early_access = 0
        self.hashes = array("Q")
        self.buckets = Counter()
        self.members = set()
        self.sort_min = self.sort_max = self.ea_min = self.ea_max = ""
    def add(self, rec, member="", ut=None):
        self.records += 1
        self.members.add(member)
        ut = get_ut(rec) if ut is None else ut
        if ut:
            self.hashes.append(ut_hash(ut))
        else:
            self.no_ut += 1
        pub = first(rec, SCHEMA["pub_info_path"])
        self.buckets[record_pub_date(pub)] += 1
        sd = get_attr(pub, SCHEMA["sortdate_attr"])
        if parse_iso(sd):
            self.sort_min = min(self.sort_min or sd, sd)
            self.sort_max = max(self.sort_max, sd)
        ea = get_attr(pub, SCHEMA["early_access_date_attr"])
        if parse_iso(ea):
            self.early_access += 1
            self.ea_min = min(self.ea_min or ea, ea)
            self.ea_max = max(self.ea_max, ea)

def save_inventory(fp, acc, issues=()):
    """Persist an inventory under OUTPUT_DIR/archive_inventory. A sample never replaces a full inventory."""
    if not fp:
        return None
    path = ARCHIVE_DIR / f"{fp[:16]}.json"
    if acc.level != "full" and path.exists():
        try:
            old = json.loads(path.read_text())
            if old.get("level") == "full" and old.get("records"):
                return None
        except Exception:
            pass
    h = np.frombuffer(acc.hashes, dtype=np.uint64) if len(acc.hashes) else np.array([], dtype=np.uint64)
    u = np.unique(h)
    if acc.level == "full":
        np.save(ARCHIVE_DIR / f"{fp[:16]}.uts.npy", u)
    data = {"content_fingerprint": fp, "level": acc.level, "inventoried_at": now_iso(),
            "reader_version": INVENTORY_READER_VERSION,
            "records": acc.records, "records_without_ut": acc.no_ut, "unique_uts": int(len(u)),
            "duplicate_uts": int(len(h) - len(u)), "members_read": len(acc.members),
            "sort_min": acc.sort_min, "sort_max": acc.sort_max, "early_access_records": acc.early_access,
            "early_access_min": acc.ea_min, "early_access_max": acc.ea_max, "issues": list(issues)[:5],
            "buckets": [[p_, v, c] for (p_, v), c in sorted(acc.buckets.items())]}
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(data))
    os.replace(tmp, path)
    return data

INVENTORY_READER_VERSION = 2          # bump when record reading changes; older SAMPLE inventories are redone

def load_inventory(fp):
    path = ARCHIVE_DIR / f"{str(fp)[:16]}.json"
    if not fp or not path.exists():
        return None
    data = json.loads(path.read_text())
    if data.get("level") != "full" and data.get("reader_version", 1) < INVENTORY_READER_VERSION:
        return None                       # sample taken with an older reader (e.g. before ProQuest unwrapping)
    data["buckets"] = Counter({(p_, v): c for p_, v, c in data["buckets"]})
    return data

def inventory_file(file_row, level="sample", sample_records=None):
    """Profile one archive. 'sample' spreads about sample_records reads evenly across all XML members."""
    sample_records = sample_records or INVENTORY_SAMPLE_RECORDS
    path = Path(file_row.path)
    size = path.stat().st_size
    acc, issues = InventoryAccumulator(level), []
    n_members = max(int(file_row.data_members or 1), 1)
    stride = max(1, n_members // sample_records) if level == "sample" else 1
    per_member = max(25, sample_records // max(n_members // stride, 1))
    pick = (lambda i, n: i % stride == 0) if stride > 1 else None   # e.g. every 20th of 1M one-record members
    try:
        for member, stream, fmt in iter_streams(path, pick):
            k = 0
            for rec in records_from_stream(path, stream, fmt, SCHEMA["record_tag"], issues, size):
                acc.add(rec, member)
                k += 1
                if level == "sample" and k >= per_member:
                    break
    except Exception as e:
        issues.append(f"{type(e).__name__}: {str(e)[:150]}")
        if acc.level == "full":
            acc.level = "partial"                 # never presented as a complete inventory
    return acc, issues

def summarize_inventory(inv):
    b = inv["buckets"]
    total = sum(b.values())
    prec, by_year, by_ym, ivs = Counter(), Counter(), Counter(), []
    for (p_, v), c in b.items():
        prec[p_] += c
        if p_ == "none":
            continue
        by_year[v[:4]] += c
        by_ym[v[:7] if p_ != "year" else f"{v}-?? (year only)"] += c
        ivs.append((bucket_interval(p_, v), p_, v))
    out = {"date_precision": " | ".join(f"{p_} {prec[p_] / total:.1%}" for p_ in PRECISION_ORDER if prec[p_])
           if total else "",
           "records_by_year": json.dumps(dict(sorted(by_year.items()))),
           "records_by_year_month": json.dumps(dict(sorted(by_ym.items())))}
    if ivs:
        lo = min(ivs, key=lambda t: t[0][0])
        hi = max(ivs, key=lambda t: t[0][1])
        out.update(min_publication_date=lo[2], min_date_precision=lo[1],
                   max_publication_date=hi[2], max_date_precision=hi[1],
                   min_publication_year=min(by_year), max_publication_year=max(by_year))
    monthly = {k: c for k, c in by_ym.items() if "year only" not in k}
    mtot = sum(monthly.values())
    if mtot:
        core = sorted(k for k, c in monthly.items() if c / mtot >= COVERAGE_MONTH_MIN_SHARE)
        inside = sum(c for k, c in monthly.items() if core[0] <= k <= core[-1])
        out["likely_coverage"] = f"{core[0]} to {core[-1]} ({inside / mtot:.1%} of month-dated records)"
    return out

def range_shares(buckets):
    """Share of dated records inside / straddling each expected range.
    Records dated only by YEAR (WoS gives them the sort date Jan 1) say nothing about which four-month chunk they
    belong to, so they are left out of the share when month- or day-dated records exist."""
    dated = [(bucket_interval(p_, v), c, p_) for (p_, v), c in buckets.items() if p_ != "none"]
    precise = [(iv, c) for iv, c, p_ in dated if p_ != "year"]
    use = precise if sum(c for _, c in precise) else [(iv, c) for iv, c, _ in dated]
    n = sum(c for _, c in use)
    res = []
    for e in EXPECTED_DF.itertuples():
        inside = boundary = 0
        for (lo, hi), c in use:
            if lo >= e.start and hi <= e.end:
                inside += c
            elif lo <= e.end and hi >= e.start:
                boundary += c
        exp = None if e.expected_records is None or pd.isna(e.expected_records) else int(e.expected_records)
        res.append((e.dataset_name, inside / n if n else 0.0, boundary / n if n else 0.0, exp))
    return n, sorted(res, key=lambda r: (-r[1], -r[2]))

def _int(v):
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return 0

def suggest_dataset(inv, true_count=0):
    """-> (suggested name, confidence, detail). Confidence: high | provisional (sample only) | ambiguous | no single match."""
    if inv is None:
        return "", "not inventoried", ""
    if EXPECTED_DF.empty:
        return "", "no expected datasets configured", ""
    n, res = range_shares(inv["buckets"])
    if n == 0:
        return "", "no dated records", ""
    best = res[0]
    second = res[1] if len(res) > 1 else ("", 0.0, 0.0, None)
    detail = "; ".join(f"{r[0]} {r[1]:.0%}" + (f" (+{r[2]:.0%} straddling)" if r[2] >= 0.005 else "")
                       for r in res if r[1] >= 0.005 or r[2] >= 0.005) or "no expected range contains these dates"
    if best[1] >= ASSIGN_MIN_SHARE_INSIDE and best[1] - second[1] >= ASSIGN_MIN_MARGIN:
        conf = "high" if inv["level"] == "full" else "provisional (sample only)"
        count = true_count or (inv["records"] if inv["level"] == "full" else 0)
        if best[3] and count:
            if abs(count - best[3]) / best[3] > 0.02:
                conf = "ambiguous"
                detail += f"; record count {count:,} differs from expected {best[3]:,}"
            else:
                detail += f"; record count matches expected ({count:,})"
        return best[0], conf, detail
    if best[1] >= 0.5:
        return best[0], "ambiguous", detail
    return "", "no single match", detail

MANIFEST_COLUMNS = [
    "content_fingerprint", "fingerprint_short", "current_file_name", "file_key", "currently_present",
    "first_seen_file_name", "first_seen_at", "file_size_bytes", "file_size", "fingerprint_method",
    "full_file_sha256", "xml_members", "non_xml_members", "non_xml_member_examples", "record_count",
    "record_count_check", "inventory_level", "members_read",
    "records_read", "records_without_ut", "unique_uts", "duplicate_uts_within_archive",
    "min_publication_date", "min_date_precision", "max_publication_date", "max_date_precision",
    "min_publication_year", "max_publication_year", "date_precision", "min_wos_sortdate", "max_wos_sortdate",
    "early_access_records", "likely_coverage", "records_by_year", "records_by_year_month", "inventoried_at",
    "inventory_issues", "citation_processing_status", "citation_completed_at", "times_citation_completed",
    "file_names_seen", "suggested_dataset", "suggestion_confidence", "suggestion_detail",
    "manual_dataset_name", "manual_expected_start", "manual_expected_end", "dataset_name",
    "dataset_assignment_source", "expected_start", "expected_end", "suggested_file_name", "notes"]
INVENTORY_SUMMARY_KEYS = ["min_publication_date", "min_date_precision", "max_publication_date", "max_date_precision",
                          "min_publication_year", "max_publication_year", "date_precision", "likely_coverage",
                          "records_by_year", "records_by_year_month"]

def load_manifest():
    m = pd.read_csv(MANIFEST_CSV, dtype=str, keep_default_na=False) if MANIFEST_CSV.exists() \
        else pd.DataFrame(columns=MANIFEST_COLUMNS)
    for c in MANIFEST_COLUMNS:
        if c not in m.columns:
            m[c] = ""
    return m

_WARNED_NAME_KEYS = set()

def _config_assignment(fp, file_name, name_to_fps=None):
    for key, val in ARCHIVE_ASSIGNMENTS.items():
        k = str(key).strip()
        if len(k) >= 12 and fp.startswith(k.lower()):
            return str(val).strip()
        if k.lower() == Path(str(file_name)).name.lower():
            if name_to_fps and len(name_to_fps.get(k.lower(), ())) > 1:
                if k not in _WARNED_NAME_KEYS:
                    warn(f"ARCHIVE_ASSIGNMENTS key {k!r} is a file name used by more than one archive "
                         "(each export downloads as wos-data.zip). It is ignored - use the fingerprint instead.")
                    _WARNED_NAME_KEYS.add(k)
                continue
            return str(val).strip()
    return ""

def _as_date_str(v):
    d = pd.to_datetime(v, errors="coerce") if str(v).strip() else pd.NaT
    return "" if pd.isna(d) else str(d.date())

def refresh_manifest():
    """Rebuild GREI_WoS_archive_manifest.csv: present files, inventories, processing status, suggestions.
    Your manual_* and notes columns are preserved; archives no longer present stay listed (currently_present = no)."""
    global MANIFEST_DF, MANIFEST_LOOKUP
    rows = {r["content_fingerprint"]: r for r in load_manifest().to_dict("records") if r["content_fingerprint"]}
    present = set()
    for r in WOS_FILES_DF.itertuples():
        fp = r.content_fingerprint
        if not fp or fp in present:            # duplicate download: the first copy represents the archive
            continue
        present.add(fp)
        row = rows.setdefault(fp, {c: "" for c in MANIFEST_COLUMNS})
        seen = [n for n in str(row.get("file_names_seen", "")).split(MULTI_SEP) if n]
        if r.file_key not in seen:
            seen.append(r.file_key)
        if not row.get("first_seen_file_name"):
            row.update(first_seen_file_name=r.file_key, first_seen_at=now_iso())
        if globals().get("FULL_SHA", {}).get(fp):
            row["full_file_sha256"] = FULL_SHA[fp]
        row.update(content_fingerprint=fp, fingerprint_short=fp[:12], current_file_name=r.file_name,
                   file_key=r.file_key, file_size_bytes=str(r.size_bytes), file_size=r.size,
                   fingerprint_method=r.fingerprint_method, xml_members=str(r.xml_members),
                   non_xml_members=str(r.non_xml_members), non_xml_member_examples=r.non_xml_member_examples,
                   file_names_seen=MULTI_SEP.join(seen))
    log = read_log()
    log = log[log.content_fingerprint != ""].reset_index().sort_values("index") if len(log) else log
    name_to_fps = defaultdict(set)
    for fp_, row_ in rows.items():
        for n in str(row_.get("file_names_seen", "")).split(MULTI_SEP):
            if n:
                name_to_fps[Path(n).name.lower()].add(fp_)
    for fp, row in rows.items():
        row["currently_present"] = "yes" if fp in present else "no"
        inv = load_inventory(fp)
        if inv:
            row.update(inventory_level=inv["level"], members_read=str(inv["members_read"]),
                       records_read=str(inv["records"]), records_without_ut=str(inv["records_without_ut"]),
                       unique_uts=str(inv["unique_uts"]) if inv["level"] == "full" else "",
                       duplicate_uts_within_archive=str(inv["duplicate_uts"]) if inv["level"] == "full" else "",
                       min_wos_sortdate=inv["sort_min"], max_wos_sortdate=inv["sort_max"],
                       early_access_records=str(inv["early_access_records"]), inventoried_at=inv["inventoried_at"],
                       inventory_issues=" | ".join(inv.get("issues", []))[:300])
            summ = summarize_inventory(inv)
            for k in INVENTORY_SUMMARY_KEYS:
                row[k] = str(summ.get(k, ""))
        g = log[log.content_fingerprint == fp] if len(log) else log
        done = g[g.status == "completed"] if len(g) else g
        # true record count = number of XML members (one record each); parsed count is the cross-check
        if _int(row.get("xml_members")):
            row["record_count"] = row["xml_members"]
        elif inv and inv["level"] == "full":
            row["record_count"] = str(inv["records"])
        if len(done) and done.record_count_check.iloc[-1]:
            row["record_count_check"] = done.record_count_check.iloc[-1]
        elif not row.get("record_count_check"):
            row["record_count_check"] = "not yet verified (verified during processing)"
        row["citation_processing_status"] = g.status.iloc[-1] if len(g) else "not processed"
        row["times_citation_completed"] = str(len(done))
        row["citation_completed_at"] = done.finished_at.iloc[-1] if len(done) else ""
        name, conf, detail = suggest_dataset(inv, _int(row.get("record_count")))
        row.update(suggested_dataset=name, suggestion_confidence=conf, suggestion_detail=detail)
        cfg = _config_assignment(fp, row.get("current_file_name", ""), name_to_fps)
        if cfg:
            if row.get("manual_dataset_name") and row["manual_dataset_name"] != cfg:
                warn(f"{row['current_file_name']}: ARCHIVE_ASSIGNMENTS says {cfg}; replacing manifest value "
                     f"{row['manual_dataset_name']}")
            row["manual_dataset_name"] = cfg
        manual = str(row.get("manual_dataset_name", "")).strip()
        if manual:
            row["dataset_name"] = manual
            row["dataset_assignment_source"] = "manual" if not (conf == "high" and name != manual) \
                else f"manual (CONFLICT: XML dates point to {name})"
        elif conf == "high":
            row["dataset_name"] = name
            row["dataset_assignment_source"] = "XML publication dates (high confidence; unconfirmed)"
        else:
            row["dataset_name"] = ""
            row["dataset_assignment_source"] = "unidentified - review" if inv else "not yet inventoried"
        exp = EXPECTED_DF[EXPECTED_DF.dataset_name == row["dataset_name"]] if row["dataset_name"] else EXPECTED_DF.iloc[0:0]
        row["expected_start"] = _as_date_str(row.get("manual_expected_start", "")) or (str(exp.start.iloc[0]) if len(exp) else "")
        row["expected_end"] = _as_date_str(row.get("manual_expected_end", "")) or (str(exp.end.iloc[0]) if len(exp) else "")
        ext = file_kind(row.get("current_file_name", "")) or ""
        sug = re.sub(r"[^\w.-]+", "_", row["dataset_name"]).strip("._") + ext if row["dataset_name"] else ""
        row["suggested_file_name"] = sug if sug and sug != row.get("current_file_name") else ""
    m = pd.DataFrame(list(rows.values()), columns=None)
    for c in MANIFEST_COLUMNS:
        if c not in m.columns:
            m[c] = ""
    m = m[MANIFEST_COLUMNS + [c for c in m.columns if c not in MANIFEST_COLUMNS]].fillna("")
    if len(m):
        m = m.sort_values(["currently_present", "min_publication_date", "current_file_name"],
                          ascending=[False, True, True])
    atomic_to_csv(m, MANIFEST_CSV)
    MANIFEST_DF = m.reset_index(drop=True)
    MANIFEST_LOOKUP = {r["content_fingerprint"]: r for r in MANIFEST_DF.to_dict("records")}
    return MANIFEST_DF

def coverage_summary_table(m=None, present_only=True):
    m = MANIFEST_DF if m is None else m
    t = m[m.currently_present == "yes"] if present_only else m
    return pd.DataFrame({
        "ZIP / file": t.current_file_name,
        "fingerprint": t.fingerprint_short,
        "records": [f"{_int(rc):,}" if _int(rc) else (f"{_int(r):,} (sample)" if r else "")
                    for rc, r in zip(t.record_count, t.records_read)],
        "count check": t.record_count_check.str.replace("not yet verified (verified during processing)", "pending"),
        "min publication date": t.min_publication_date,
        "max publication date": t.max_publication_date,
        "unique UTs": t.unique_uts,
        "likely coverage": t.likely_coverage,
        "suggested dataset": (t.suggested_dataset + " [" + t.suggestion_confidence + "]").str.strip(),
        "assigned dataset": t.dataset_name,
        "basis": t.dataset_assignment_source,
        "citation processing": t.citation_processing_status,
    }).reset_index(drop=True)

if len(EXPECTED_DF):
    print("Expected TDM Studio datasets (from configuration):")
    display(EXPECTED_DF)

In [ ]:
# ---- Profile new archives (read-only) and update the manifest ----------------------------
if WOS_FILES_DF.empty:
    print("No WoS files to inventory.")
else:
    _files = WOS_FILES_DF.drop_duplicates("content_fingerprint")
    _levels = {fp: (load_inventory(fp) or {}).get("level", "") for fp in _files.content_fingerprint}
    if PRE_INVENTORY_MODE == "off":
        todo = _files.iloc[0:0]
        print("PRE_INVENTORY_MODE = 'off': archives will be inventoried during the citation run only.")
    elif PRE_INVENTORY_MODE == "full":
        todo = _files[_files.content_fingerprint.map(_levels) != "full"]
    else:
        todo = _files[~_files.content_fingerprint.map(_levels).isin(["full", "sample"])]
    sample_n = INVENTORY_SAMPLE_RECORDS
    if TEST_MODE and len(todo):                 # first validation run: profile only the test archive, lightly
        fps_done, legacy_done = completed_status()
        pending_first = pd.concat([todo[~todo.apply(lambda r: is_completed(r, fps_done, legacy_done), axis=1)],
                                   todo[todo.apply(lambda r: is_completed(r, fps_done, legacy_done), axis=1)]])
        todo = pending_first.head(TEST_MAX_FILES)
        sample_n = min(INVENTORY_SAMPLE_RECORDS, TEST_MAX_RECORDS)
    if len(todo):
        print(f"Profiling {len(todo)} archive(s) ({PRE_INVENTORY_MODE}, ~{sample_n:,} records); "
              "source files are only read...")
    for r in todo.itertuples():
        t0 = time.time()
        acc, issues = inventory_file(r, "full" if PRE_INVENTORY_MODE == "full" else "sample", sample_n)
        save_inventory(r.content_fingerprint, acc, issues)
        print(f"   {r.file_key}: {acc.records:,} records read from {len(acc.members)} of {r.data_members} member(s) "
              f"in {time.time() - t0:,.1f}s" + (f" | issues: {issues[0][:100]}" if issues else ""))
    refresh_manifest()
    print(f"\nArchive manifest: {MANIFEST_CSV}")
    display(coverage_summary_table())
    unident = MANIFEST_DF[(MANIFEST_DF.currently_present == "yes") & (MANIFEST_DF.dataset_name == "")
                          & (MANIFEST_DF.inventory_level != "")]
    prov = unident[unident.suggestion_confidence.str.startswith("provisional")]
    review = unident.drop(prov.index)
    for _, r in prov.iterrows():
        print(f"ℹ️  {r.current_file_name}: provisionally matches {r.suggested_dataset} (sample: {r.suggestion_detail}). "
              "Confirmed automatically by the full inventory during processing, or assign it yourself.")
    for _, r in review.iterrows():
        warn(f"{r.current_file_name}: not identified ({r.suggestion_confidence}). Evidence: {r.suggestion_detail}")
    conflicts = MANIFEST_DF[MANIFEST_DF.dataset_assignment_source.str.contains("CONFLICT")]
    for _, r in conflicts.iterrows():
        warn(f"{r.current_file_name}: {r.dataset_assignment_source}")
    if len(review):
        print("   -> If you know which dataset these are, add them to ARCHIVE_ASSIGNMENTS (or the manifest's "
              "manual_dataset_name column). Processing does NOT depend on identification.")

### 2h. Matching rules (definitions only — nothing is processed yet)

**Scope.** This notebook identifies and preserves citation relationships. For each citing WoS publication it stores the
structured WoS values and raw text *as given* — no institution, country, funder or subject normalization — so a separate
analysis notebook can clean and analyse them later. Multi-valued fields use ` | ` between values (full addresses ` || `).

| Group | Columns in the edge files |
|---|---|
| identity | `citing_wos_ut`, `citing_doi`, `citing_title`, `citing_year`, `citing_publication_date`, `citing_source`, `citing_issn`, `citing_eissn`, `citing_document_type`, `citing_pubtype` |
| people & places | `citing_authors`, `citing_author_address_links`, `citing_affiliations_raw`, `citing_organizations` (WoS preferred names), `citing_organizations_original`, `citing_cities`, `citing_regions`, `citing_countries` |
| corresponding author | `citing_corresponding_authors`, `citing_corresponding_address_raw`, `citing_corresponding_organizations`, `citing_corresponding_countries` |
| subject | `citing_wos_categories`, `citing_research_areas`, `citing_author_keywords`, `citing_keywords_plus`, `citing_abstract` |
| funding | `citing_funding_agencies_original`, `citing_funding_agencies_preferred`, `citing_funding_agencies_parent`, `citing_grant_numbers`, `citing_funding_text`, `citing_funding_grants_json` (every grant verbatim) |

Metadata is only extracted for the few records that cite a target DOI, so the extra fields do not slow the scan.
For each WoS record: read every cited reference → extract and normalize its DOI(s) → check exact membership in the target
DOI set. Citing-document metadata is extracted **only** when a record has at least one match (much faster).

Relationship key: `target_doi_normalized || citing_doc_key`, where `citing_doc_key` is the WoS UT, or `DOI:<citing doi>` if
the UT is missing, or (last resort) `NOKEY:<file>#<member>#<record number>`. A target cited several times in one reference
list produces **one** relationship.

In [ ]:
def match_record(rec, target_set):
    """Return (n_refs, n_refs_with_doi, matches, malformed, near_misses).
    matches = {target_doi: {"n": mentions, "raw_doi": str, "method": str, "ref": str}}"""
    refs = get_refs(rec)
    n_doi, matches, malformed, near = 0, {}, [], []
    for ref in refs:
        found, bad = ref_dois(ref)
        malformed.extend(bad)
        if not found:
            continue
        n_doi += 1
        for d, (method, raw) in found.items():
            if d in target_set:
                m = matches.get(d)
                if m is None:
                    matches[d] = {"n": 1, "raw_doi": raw, "method": method, "ref": ref_summary(ref)}
                else:
                    m["n"] += 1
            elif d.split("/", 1)[0] in TARGET_PREFIXES:
                vm = VERSION_SUFFIX_RE.match(d)
                cands = set()
                if vm and vm.group(1) in target_set:
                    cands.add(vm.group(1))
                cands |= TARGET_BASE_TO_TARGETS.get(d, set())
                for c in cands:
                    near.append((d, c, ref_summary(ref, 200)))
    return len(refs), n_doi, matches, malformed, near

EDGE_COLUMNS = [
    "relationship_key", "target_doi_normalized", "citing_doc_key", "citing_key_type",
    "citing_wos_ut", "citing_proquest_goid", "citing_doi", "citing_title", "citing_year", "citing_publication_date",
    "citing_source", "citing_document_type", "citing_pubtype", "citing_authors", "citing_author_address_links",
    "citing_affiliations_raw", "citing_organizations", "citing_cities", "citing_regions", "citing_countries",
    "citing_organizations_original", "citing_corresponding_authors", "citing_corresponding_address_raw",
    "citing_corresponding_organizations", "citing_corresponding_countries",
    "citing_issn", "citing_eissn", "citing_wos_categories", "citing_research_areas",
    "citing_author_keywords", "citing_keywords_plus", "citing_abstract",
    "citing_funding_agencies_original", "citing_funding_agencies_preferred", "citing_funding_agencies_parent",
    "citing_grant_numbers", "citing_funding_text", "citing_funding_grants_json",
    "matched_cited_reference_doi", "matched_cited_reference_raw", "match_method",
    "n_mentions_in_reference_list", "citing_reference_count",
    "source_wos_file", "source_wos_member", "source_wos_dataset", "source_record_number",
    "run_id", "processing_datetime", "source_archive_fingerprint",
]

def build_edges(rec, matches, n_refs, file_key, member, dataset, rec_no, run_id, fingerprint=""):
    meta = extract_citing(rec)
    if meta["citing_wos_ut"]:
        ck, ktype = meta["citing_wos_ut"], "UT"
    elif meta["citing_doi"]:
        ck, ktype = "DOI:" + meta["citing_doi"], "citing_doi"
    else:
        ck, ktype = f"NOKEY:{file_key}#{member}#{rec_no}", "file_position"
    ts = now_iso()
    rows = []
    for tdoi, m in matches.items():
        row = {"relationship_key": f"{tdoi}||{ck}", "target_doi_normalized": tdoi,
               "citing_doc_key": ck, "citing_key_type": ktype, **meta,
               "matched_cited_reference_doi": m["raw_doi"], "matched_cited_reference_raw": m["ref"],
               "match_method": m["method"], "n_mentions_in_reference_list": m["n"],
               "citing_reference_count": n_refs, "source_wos_file": file_key, "source_wos_member": member,
               "source_wos_dataset": dataset, "source_record_number": rec_no,
               "run_id": run_id, "processing_datetime": ts, "source_archive_fingerprint": fingerprint}
        rows.append({c: row.get(c, "") for c in EDGE_COLUMNS})
    return rows

# ---- prove the matching logic on a real sample record by injecting a synthetic reference ----------
def injected_match_selftest():
    if not (globals().get("SAMPLE_RECS") and TARGET_DOI_SET):
        print("Skipped (no sample records or no target DOIs).")
        return False
    rec = next((r for r in SAMPLE_RECS if get_refs(r)), None)
    if rec is None:
        warn("No sample record has cited references; cannot run the injection test.")
        return False
    rec = copy.deepcopy(rec)
    tgt = sorted(TARGET_DOI_SET)[0]
    first_ref = get_refs(rec)[0]
    parent = first_ref.getparent() if hasattr(first_ref, "getparent") else next(
        p_ for p_ in rec.iter() if first_ref in list(p_))
    ns = first_ref.tag[: first_ref.tag.index("}") + 1] if "}" in first_ref.tag else ""
    new_ref = parent.makeelement(first_ref.tag, {})
    doi_el = new_ref.makeelement(ns + SCHEMA["ref_doi_tags"][0], {})
    doi_el.text = "https://doi.org/" + tgt.upper() + "."      # deliberately messy
    new_ref.append(doi_el)
    parent.append(new_ref)
    _, _, matches, _, _ = match_record(rec, TARGET_DOI_SET)
    ok = tgt in matches
    print(("✅" if ok else "❌") + f" Injected reference to {tgt} {'was' if ok else 'was NOT'} matched "
          f"(sample record {get_ut(rec) or '?'}; the real file is not changed).")
    return ok

INJECTION_TEST_OK = injected_match_selftest()

### 2i. Processing machinery (definitions only)

* **Processing log** (`GREI_WoS_processing_log.csv`) is *append-only*: a `started` row when a file begins and a
  `completed` / `failed` / `interrupted` row when it ends. A file counts as done only if its latest row is `completed`
  **and** its size is unchanged.
* **Matches** are appended to `GREI_WoS_citation_edges_raw_appendonly.csv` every `CHECKPOINT_EVERY_RECORDS` records and at
  the end of each file, so an interrupted kernel loses little. An interrupted file is simply re-read next time; any
  duplicate rows this creates are removed when the master is built.
* **Master** (`GREI_WoS_citation_edges_master.csv`) is rebuilt from the raw file and deduplicated on
  `relationship_key`, so overlapping datasets, reruns and processing order never inflate counts.
* Source WoS files are only opened for reading.

In [ ]:
def _matches_any(row, patterns):
    return any(fnmatch.fnmatch(row.file_key, p) or fnmatch.fnmatch(row.file_name, p) or row.file_key == p
               or (len(p) >= 12 and row.content_fingerprint.startswith(p)) for p in patterns)

def select_files(test=False):
    df = WOS_FILES_DF.copy()
    if PROCESS_FILES == "selected":
        df = df[df.apply(lambda r: _matches_any(r, SELECTED_FILES), axis=1)]
        if df.empty:
            warn(f"No files matched SELECTED_FILES = {SELECTED_FILES}")
    same = df[df.content_fingerprint.duplicated()]
    for _, r in same.iterrows():
        twin = df[df.content_fingerprint == r.content_fingerprint].iloc[0]
        warn(f"{r.file_key} has identical content to {twin.file_key} (duplicate download) -> skipped")
    df = df[~df.content_fingerprint.duplicated()]
    if test:
        fps, legacy = completed_status()
        done = df.apply(lambda r: is_completed(r, fps, legacy), axis=1) if len(df) else pd.Series(dtype=bool)
        return pd.concat([df[~done], df[done]]).head(TEST_MAX_FILES), pd.DataFrame()
    skipped = pd.DataFrame()
    if SKIP_COMPLETED_FILES and len(df):
        fps, legacy = completed_status()
        log = read_log()
        prev_fp = dict(zip(log[log.status == "completed"].file_key, log[log.status == "completed"].content_fingerprint))
        done = df.apply(lambda r: is_completed(r, fps, legacy), axis=1)
        for _, r in df[~done & df.file_key.isin(prev_fp)].iterrows():
            warn(f"{r.file_key}: a file with this name was processed before but its CONTENT is different "
                 "(e.g. a new download that reused a generic name) -> it will be processed.")
        forced = df.apply(lambda r: _matches_any(r, FORCE_REPROCESS_FILES), axis=1)
        skipped = df[done & ~forced]
        df = df[~done | forced]
    return df.reset_index(drop=True), skipped

class _Progress:
    def __init__(self, desc):
        self.desc, self.n, self.t0 = desc, 0, time.time()
        self.bar = tqdm(desc=desc, unit=" rec", mininterval=5, smoothing=0.05) if HAVE_TQDM else None
    def update(self, k=1, matches=0):
        self.n += k
        if self.bar is not None:
            self.bar.update(k)
            if self.n % 50_000 == 0:
                self.bar.set_postfix(relationships=matches)
        elif VERBOSE and self.n % PROGRESS_EVERY_RECORDS == 0:
            rate = self.n / max(time.time() - self.t0, 1e-9)
            print(f"   {self.desc}: {self.n:,} records | {rate:,.0f} rec/s | relationships so far {matches:,}", flush=True)
    def close(self):
        if self.bar is not None:
            self.bar.close()

def dataset_label(file_row):
    # TDM Studio dataset name if known (manual or confident identification), else the folder name
    return MANIFEST_LOOKUP.get(file_row.content_fingerprint, {}).get("dataset_name") or file_row.dataset

def process_file(file_row, target_set, *, test=False, run_id=""):
    # Stream one WoS file/archive, append matches to the raw edge file, and return a stats dict.
    # In a real (non-test) run the same pass also builds the archive's full inventory (UTs, publication dates).
    P = output_paths(test)
    path, file_key = Path(file_row.path), file_row.file_key
    fp = file_row.content_fingerprint
    dataset = dataset_label(file_row)
    st = {c: 0 for c in ["records_examined", "records_prefiltered", "cited_refs_examined", "cited_refs_with_doi", "raw_reference_matches",
                         "records_missing_ut", "malformed_ref_dois", "version_near_misses"]}
    is_zip = file_row.file_name.lower().endswith(".zip")
    n_xml = int(file_row.xml_members or 0) if is_zip else ""
    st.update(xml_members=n_xml, xml_members_read=0, record_count_check="",
              file_key=file_key, file_name=file_row.file_name, dataset=dataset, content_fingerprint=fp,
              file_size_bytes=int(file_row.size_bytes), run_id=run_id, started_at=now_iso(),
              status="started", error="", warnings="",
              settings=json.dumps({"scan_ref_text": SCAN_ALL_REFERENCE_TEXT_FOR_DOIS, "test": test,
                                   "record_tag": SCHEMA["record_tag"], "refs_paths": SCHEMA["refs_paths"]}))
    if not test:
        append_csv(P["log"], [st], LOG_COLUMNS)          # 'started' row: proves an attempt even if the kernel dies
    rel_keys, tdois, buffer, issues = set(), set(), [], []
    malformed_rows, near_rows = [], []
    inv = None if test else InventoryAccumulator("full")
    max_rec = TEST_MAX_RECORDS if test else None
    stop_after = TEST_STOP_AFTER_MATCHES if test else None
    prog = _Progress(file_row.file_name[:40])
    t0 = time.time()
    last_member = None

    def flush():
        append_csv(P["edges_raw"], buffer, EDGE_COLUMNS)
        buffer.clear()
        append_csv(P["malformed"], malformed_rows, ["file_key", "citing_wos_ut", "raw_value"])
        malformed_rows.clear()
        append_csv(P["nearmiss"], near_rows, ["file_key", "citing_wos_ut", "cited_doi", "target_doi", "reference"])
        near_rows.clear()

    try:
        prefilter = ([f"{x}/".encode() for x in sorted(TARGET_PREFIXES)]
                     if (FAST_PREFILTER and not test and TARGET_PREFIXES) else None)
        for rec_no, (member, rec) in enumerate(iter_records(path, SCHEMA["record_tag"], issues, prefilter), 1):
            st["records_examined"] += 1
            if member != last_member:
                st["xml_members_read"] += 1
                last_member = member
            ut = get_ut(rec)
            if inv is not None:
                inv.add(rec, member, ut)
            if rec.get("prefiltered") == "1":         # contains no target DOI prefix: cannot match
                st["records_prefiltered"] += 1
                prog.update(1, len(rel_keys))
                if st["records_examined"] % CHECKPOINT_EVERY_RECORDS == 0:
                    flush()
                if max_rec and st["records_examined"] >= max_rec:
                    break
                continue
            n_refs, n_doi, matches, malformed, near = match_record(rec, target_set)
            st["cited_refs_examined"] += n_refs
            st["cited_refs_with_doi"] += n_doi
            if malformed:
                st["malformed_ref_dois"] += len(malformed)
                if len(malformed_rows) < 25 and st["malformed_ref_dois"] <= 25:
                    malformed_rows.extend({"file_key": file_key, "citing_wos_ut": ut, "raw_value": m[:300]}
                                          for m in malformed[:5])
            if near:
                st["version_near_misses"] += len(near)
                near_rows.extend({"file_key": file_key, "citing_wos_ut": ut, "cited_doi": d,
                                  "target_doi": t, "reference": r} for d, t, r in near)
            if matches:
                rows = build_edges(rec, matches, n_refs, file_key, member, dataset, rec_no, run_id, fp)
                st["raw_reference_matches"] += sum(m["n"] for m in matches.values())
                if rows[0]["citing_key_type"] != "UT":
                    st["records_missing_ut"] += 1
                for r in rows:
                    rel_keys.add(r["relationship_key"])
                    tdois.add(r["target_doi_normalized"])
                buffer.extend(rows)
            prog.update(1, len(rel_keys))
            if st["records_examined"] % CHECKPOINT_EVERY_RECORDS == 0:
                flush()
            if max_rec and st["records_examined"] >= max_rec:
                break
            if stop_after and len(rel_keys) >= stop_after:
                break
        flush()
        st["status"] = "completed"
    except KeyboardInterrupt:
        flush()
        st["status"] = "interrupted"
        st["error"] = "KeyboardInterrupt (kernel interrupted)"
        raise
    except Exception as e:
        flush()
        st["status"] = "failed"
        st["error"] = f"{type(e).__name__}: {str(e)[:500]}"
        if VERBOSE:
            traceback.print_exc(limit=3)
    finally:
        prog.close()
        st["finished_at"] = now_iso()
        st["seconds"] = round(time.time() - t0, 1)
        st["unique_relationships"] = len(rel_keys)
        st["unique_target_dois_matched"] = len(tdois)
        if issues:
            st["warnings"] = f"{len(issues)} parse issue(s); first: " + " | ".join(issues[:3])
            warn(f"{file_key}: {st['warnings'][:300]}")
        if st["records_examined"] == 0 and st.get("status") == "completed":
            st["status"] = "failed"
            st["error"] = f"No records with tag {SCHEMA['record_tag']!r} found"
        if is_zip and st["status"] == "completed" and not test:
            if st["records_examined"] == n_xml and st["xml_members_read"] == n_xml:
                st["record_count_check"] = "OK: parsed records = XML members"
            else:
                st["record_count_check"] = (f"MISMATCH: {st['records_examined']:,} records parsed from "
                                            f"{st['xml_members_read']:,} of {n_xml:,} XML members")
                warn(f"{file_key}: {st['record_count_check']} (XML member count is the true record count)")
        if inv is not None and st["status"] == "completed":
            save_inventory(fp, inv, issues)               # full archive inventory from this same pass
        if not test and st["status"] == "completed" and rel_keys:
            try:                                          # results reach the persistent master before the
                build_master(test=False, write_xlsx=False)    # log says "completed"
            except Exception as e:
                st["status"] = "failed"
                st["error"] = f"matches saved to the raw journal, but updating the master failed: {e}"
        if not test:
            append_csv(P["log"], [st], LOG_COLUMNS)
    return st

for _p in [output_paths(False)["edges_raw"], output_paths(True)["edges_raw"]]:
    ensure_csv_columns(_p, EDGE_COLUMNS)                  # upgrade files written by an older notebook version

In [ ]:
# ----- build the deduplicated master edge table ------------------------------------------------
TARGET_META_COLUMNS = [c for c in [
    TARGET_DOI_COLUMN, "attributes_doi", "attributes_titles", "attributes_publicationYear", "attributes_publisher",
    "relationships_client_data_id", "attributes_types_resourceTypeGeneral", "attributes_types_resourceType",
    "attributes_version", "attributes_container_title", "attributes_creators", "attributes_url",
    "attributes_contentUrl", "attributes_citationCount"] if c in targets_df.columns]

def first_title(v):
    """Pull a readable title out of DataCite's titles field (JSON / Python-list text or plain text)."""
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return ""
    s = str(v).strip()
    for loader in (json.loads, ast.literal_eval):
        try:
            obj = loader(s)
            if isinstance(obj, dict):
                obj = [obj]
            if isinstance(obj, list):
                for o in obj:
                    if isinstance(o, dict) and o.get("title"):
                        return str(o["title"])
                    if isinstance(o, str) and o:
                        return o
        except Exception:
            pass
    return s

targets_df["target_title"] = targets_df["attributes_titles"].map(first_title) if "attributes_titles" in targets_df else ""

def target_meta_table():
    """One row of target metadata per normalized DOI (first workbook row), plus how many rows share the DOI."""
    t = targets_df[targets_df.target_doi_normalized.notna()]
    meta = t.drop_duplicates("target_doi_normalized")[["target_doi_normalized", "target_title"] + TARGET_META_COLUMNS]
    meta = meta.rename(columns={TARGET_DOI_COLUMN: "target_doi"})
    n = t.groupby("target_doi_normalized").size().rename("n_workbook_rows_with_doi")
    rows = t.groupby("target_doi_normalized")["source_excel_row"].apply(lambda s: ",".join(map(str, s))).rename("workbook_rows")
    return meta.merge(n, on="target_doi_normalized").merge(rows, on="target_doi_normalized")

PROV_COLS = ["all_source_wos_files", "all_source_wos_datasets", "all_source_archive_fingerprints"]

def unescape_text(v):
    """Undo HTML escaping (e.g. '&amp;' -> '&'); TDM Studio escapes the embedded WoS text twice."""
    if not isinstance(v, str) or "&" not in v:
        return v
    for _ in range(3):
        u = html.unescape(v)
        if u == v:
            break
        v = u
    return v

def build_master(test=False, write_xlsx=True):
    """Merge new matches into the PERSISTENT master and deduplicate on relationship_key.
    The existing master is the accumulated state from all earlier runs (it is never rebuilt from scratch),
    and the raw append-only journal adds whatever is new. Either file alone is enough to recover from:
    deleting the journal after a successful run loses nothing."""
    P = output_paths(test)
    frames, n_before = [], 0
    if P["master_csv"].exists():
        old = pd.read_csv(P["master_csv"], dtype=str, keep_default_na=False)
        n_before = len(old)
        for c in EDGE_COLUMNS + PROV_COLS:
            if c not in old.columns:
                old[c] = ""
        for agg_col, single in [("all_source_wos_files", "source_wos_file"),
                                ("all_source_wos_datasets", "source_wos_dataset"),
                                ("all_source_archive_fingerprints", "source_archive_fingerprint")]:
            old[agg_col] = old[agg_col].where(old[agg_col] != "", old[single])
        old = old[EDGE_COLUMNS + PROV_COLS].assign(_order=0)
        frames.append(old)
    if P["edges_raw"].exists():
        raw = pd.read_csv(P["edges_raw"], dtype=str, keep_default_na=False)
        for c in EDGE_COLUMNS:
            if c not in raw.columns:
                raw[c] = ""
        raw = raw[EDGE_COLUMNS].assign(all_source_wos_files=raw.source_wos_file,
                                       all_source_wos_datasets=raw.source_wos_dataset,
                                       all_source_archive_fingerprints=raw.source_archive_fingerprint, _order=1)
        frames.append(raw)
    if not frames:
        if VERBOSE:
            print("No citation matches recorded yet.")
        return pd.DataFrame(columns=["relationship_key", "target_doi_normalized", "citing_doc_key"] + EDGE_COLUMNS)
    allr = pd.concat(frames, ignore_index=True)
    allr = allr.sort_values(["_order", "processing_datetime", "source_wos_file", "source_record_number"], kind="stable")
    prov = allr.groupby("relationship_key")[PROV_COLS].agg(
        lambda col: MULTI_SEP.join(sorted(set().union(*map(_split_multi_set, col))))).reset_index()
    # one row per relationship: first-seen values win, but blanks are filled from later rows, so
    # reprocessing an archive can add fields introduced by a newer notebook version (backfill)
    vals = allr[EDGE_COLUMNS].replace("", np.nan)
    master = vals.groupby("relationship_key", sort=False).first().reset_index().fillna("")[EDGE_COLUMNS] \
        .merge(prov, on="relationship_key", how="left")
    master["n_source_files_with_relationship"] = master.all_source_wos_files.map(lambda v: len(_split_multi_set(v)))
    master = master.merge(target_meta_table(), on="target_doi_normalized", how="left")
    lk = lambda f, k: MANIFEST_LOOKUP.get(f, {}).get(k, "")
    master["archive_dataset_name"] = master.source_archive_fingerprint.map(lambda f: lk(f, "dataset_name"))
    master["archive_current_file_name"] = master.source_archive_fingerprint.map(lambda f: lk(f, "current_file_name"))
    master["all_archive_dataset_names"] = master.all_source_archive_fingerprints.map(
        lambda v: MULTI_SEP.join(sorted({lk(f, "dataset_name") or lk(f, "current_file_name") or f[:12]
                                         for f in _split_multi_set(v)})))
    front = ["relationship_key", "target_doi", "target_doi_normalized", "target_title"]
    tmeta = [c for c in TARGET_META_COLUMNS if c != TARGET_DOI_COLUMN] + ["n_workbook_rows_with_doi", "workbook_rows"]
    ordered = front + tmeta + [c for c in EDGE_COLUMNS if c not in front] + \
              ["n_source_files_with_relationship"] + PROV_COLS + \
              ["archive_dataset_name", "archive_current_file_name", "all_archive_dataset_names"]
    master = master[[c for c in ordered if c in master.columns]].fillna("")
    for c in master.columns:                     # '&amp;' -> '&' (fixes rows saved by earlier runs too)
        if c.startswith("citing_") or c in ("matched_cited_reference_raw", "matched_cited_reference_doi"):
            master[c] = master[c].map(unescape_text)
    master = master.sort_values(["target_doi_normalized", "citing_year", "citing_doc_key"]).reset_index(drop=True)
    atomic_to_csv(master, P["master_csv"])
    if write_xlsx:
        write_excel({"citation_edges": master}, P["master_xlsx"])
    if VERBOSE:
        print(f"Master updated: {n_before:,} existing + {len(master) - n_before:,} new = {len(master):,} unique "
              f"relationships -> {P['master_csv'].name}")
    return master

def write_excel(sheets, path):
    """Write DataFrames to .xlsx with frozen header, filters, and Excel-safe text (limits/illegal characters)."""
    if not HAVE_OPENPYXL:
        warn(f"openpyxl missing; skipped {Path(path).name}")
        return
    truncated = 0
    def safe(v):
        nonlocal truncated
        if isinstance(v, str):
            v = ILLEGAL_CHARACTERS_RE.sub("", v)
            if len(v) > 32_000:
                truncated += 1
                v = v[:32_000] + " …[truncated in Excel; full text in CSV]"
        return v
    tmp = Path(str(path) + ".tmp.xlsx")
    with pd.ExcelWriter(tmp, engine="openpyxl") as xw:
        for name, df in sheets.items():
            out = df.copy()
            for c in out.columns:
                if out[c].dtype == object or pd.api.types.is_string_dtype(out[c]):
                    out[c] = out[c].map(safe)
            out.to_excel(xw, sheet_name=name[:31], index=False)
            ws = xw.sheets[name[:31]]
            ws.freeze_panes = "A2"
            if len(out.columns) and len(out):
                ws.auto_filter.ref = ws.dimensions
            for i, c in enumerate(out.columns, 1):
                ws.column_dimensions[openpyxl.utils.get_column_letter(i)].width = 45 if "title" in str(c) else 18
    _replace_with_retry(tmp, path)
    if truncated:
        warn(f"{truncated} very long cell(s) were shortened in {Path(path).name} (CSV copy is complete).")

## 3. Process the export
### 3a. TEST — runs only when `TEST_MODE = True`
The first validation run reads **one ZIP** (an unprocessed one if available) and about **5,000 records**
(`TEST_MAX_FILES = 1`, `TEST_MAX_RECORDS = 5_000`). It writes `TEST_*` outputs only and **does not** write to the processing
log or the real master, so the archive is still "new" for the full run. Review every check before switching `TEST_MODE`
to `False`.

A 5,000-record sample will often contain **no** real citation of a target DOI. That is expected: check 4b then shows `WARN`,
not `FAIL`, and the test still passes, because check 4a proves matching end-to-end by injecting a reference to a real
target DOI (written in messy form: URL prefix, upper case, trailing period) into a copy of a real WoS record from this
archive. The archive itself is never modified. Raise `TEST_MAX_RECORDS` if you want to look for real matches.

In [ ]:
if TEST_MODE:
    TP = output_paths(test=True)
    for k in ["edges_raw", "master_csv", "malformed", "nearmiss"]:
        if TP[k].exists():
            TP[k].unlink()                                   # only TEST_ output files are removed
    test_files, _ = select_files(test=True)
    print(f"TEST MODE: {len(test_files)} file(s), ≤{TEST_MAX_RECORDS:,} records each\n")
TEST_OK = None
if TEST_MODE and test_files.empty:
    print("No WoS export is present to test. Download wos-data.zip into WOS_DATA_DIR and run again.")
    TEST_MASTER = pd.DataFrame()
elif TEST_MODE:
    test_stats = [process_file(r, TARGET_DOI_SET, test=True, run_id="TEST") for r in test_files.itertuples()]
    TEST_STATS_DF = pd.DataFrame(test_stats)
    display(TEST_STATS_DF[["file_key", "status", "records_examined", "cited_refs_examined", "cited_refs_with_doi",
                           "raw_reference_matches", "unique_relationships", "unique_target_dois_matched",
                           "malformed_ref_dois", "version_near_misses", "seconds", "error"]])
    TEST_MASTER = build_master(test=True, write_xlsx=False)

    def check(label, ok, warn_only=False, detail=""):
        status = "PASS" if ok else ("WARN" if warn_only else "FAIL")
        print(f"[{status}] {label}" + (f" — {detail}" if detail else ""))
        return ok or warn_only

    s = TEST_STATS_DF
    recs = int(s.records_examined.sum())
    rate = recs / max(float(s.seconds.sum()), 0.1)
    print()
    results = [
        check("1. WoS parser reads records", recs > 0 and (s.status == "completed").all(), detail=f"{recs:,} records, {rate:,.0f} rec/s"),
        check("2. Cited-reference lists are found", s.cited_refs_examined.sum() > 0, detail=f"{int(s.cited_refs_examined.sum()):,} references"),
        check("3. Cited-reference DOIs are extracted", s.cited_refs_with_doi.sum() > 0, detail=f"{int(s.cited_refs_with_doi.sum()):,} references with a DOI"),
        check("4a. Matching works on an injected reference", bool(INJECTION_TEST_OK)),
        check("4b. Real matches to target DOIs found in the test sample", len(TEST_MASTER) > 0, warn_only=True,
              detail=f"{len(TEST_MASTER)} relationships" if len(TEST_MASTER) else
              f"none in this {int(s.records_examined.sum()):,}-record sample - expected; check 4a covers matching"),
    ]
    if len(TEST_MASTER):
        m = TEST_MASTER
        pct = lambda col: f"{100 * (m[col] != '').mean():.0f}% of matched rows"
        ut_share = (m.citing_wos_ut != "").mean()
        core_share = min((m[c] != "").mean() for c in ["citing_year", "citing_title", "citing_source"])
        results += [
            check("5. Citing WoS UT captured", ut_share == 1, warn_only=ut_share >= 0.9, detail=pct("citing_wos_ut")),
            check("6. Citing DOI / year / title / source captured", core_share == 1, warn_only=core_share >= 0.9,
                  detail=f"DOI {pct('citing_doi')}, year {pct('citing_year')}, title {pct('citing_title')}, source {pct('citing_source')}"),
            check("7. Affiliations captured", (m.citing_affiliations_raw != "").any(), warn_only=True,
                  detail=f"raw {pct('citing_affiliations_raw')}, countries {pct('citing_countries')}"),
            check("8. Edge table has one row per target × citing document", not m.relationship_key.duplicated().any()),
        ]
        ext = ["citing_corresponding_authors", "citing_wos_categories", "citing_research_areas",
               "citing_author_keywords", "citing_keywords_plus", "citing_abstract", "citing_issn", "citing_eissn",
               "citing_funding_agencies_original", "citing_funding_agencies_preferred", "citing_grant_numbers",
               "citing_funding_text"]
        cov = {c.replace("citing_", ""): f"{100 * (m[c] != '').mean():.0f}%" for c in ext if c in m}
        check("9. Extended metadata captured (coverage among matched rows; blanks can be genuine)",
              any(v != "0%" for v in cov.values()), warn_only=True, detail=", ".join(f"{k} {v}" for k, v in cov.items()))
        print("\nExamples of matched citation relationships:")
        display(m[["target_doi_normalized", "target_title", "citing_wos_ut", "citing_doi", "citing_year",
                   "citing_title", "citing_source", "citing_countries", "matched_cited_reference_doi",
                   "match_method", "n_mentions_in_reference_list"]].head(10))
    elif SCHEMA_COVERAGE is not None and SHOW_DETAILS:
        print("\nField capture checked on the sample records instead (section 2f):")
        display(SCHEMA_COVERAGE)
    TEST_OK = all(results)
    print("\n" + ("✅ TEST PASSED." if TEST_OK else "❌ TEST FAILED.") + " See section 5 at the bottom for what to do next.")
else:
    print("TEST_MODE is False — test step skipped.")

### 3b. FULL RUN — runs only when `TEST_MODE = False`
Runs only when `TEST_MODE = False`. Completed files are skipped (unless listed in `FORCE_REPROCESS_FILES`).
You can stop the kernel at any time; finished files stay finished and partial results are kept.

In [ ]:
SAFE_TO_REMOVE_MSG = ("This archive has been processed successfully and its results are saved. "
                      "It can now be deleted from your Downloads folder.")
RUN_STATS, RUN_SKIPPED, RUN_ID = [], pd.DataFrame(), ""

def backup_state(run_id):
    """Copy the small persistent state files before this run changes them (keeps the last KEEP_STATE_BACKUPS)."""
    if not KEEP_STATE_BACKUPS:
        return
    d = OUTPUT_DIR / "state_backups" / run_id
    files = [output_paths(False)[k] for k in ["log", "master_csv", "edges_raw"]] + [MANIFEST_CSV]
    files = [f for f in files if f.exists()]
    if not files:
        return
    d.mkdir(parents=True, exist_ok=True)
    for f in files:
        shutil.copy2(f, d / f.name)
    for old in sorted((OUTPUT_DIR / "state_backups").iterdir())[:-KEEP_STATE_BACKUPS]:
        shutil.rmtree(old, ignore_errors=True)
    print(f"State backed up to {d.relative_to(OUTPUT_DIR)} ({len(files)} small files)")

if TEST_MODE:
    print("TEST_MODE is True — the full run is skipped. Set TEST_MODE = False when the test passes.")
else:
    RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")
if not TEST_MODE and not STATE_OK and not IGNORE_STATE_WARNINGS:
    print("❌ FULL RUN STOPPED: the persistent state is incomplete (see the start-of-run integrity check).")
    print("   Restore with restore_latest_backup(), or set IGNORE_STATE_WARNINGS = True to continue anyway.")
elif not TEST_MODE:
    backup_state(RUN_ID)
    to_do, RUN_SKIPPED = select_files(test=False)
    print(f"Run {RUN_ID}: {len(to_do):,} archive(s) to process ({human_size(to_do.size_bytes.sum())}); "
          f"{len(RUN_SKIPPED):,} present but already completed (skipped).")
    if to_do.empty:
        print("Nothing new to process. Download the next WoS export and run the notebook again.")
    run_stats = RUN_STATS
    t_run = time.time()
    try:
        for i, r in enumerate(to_do.itertuples(), 1):
            print(f"\n[{i}/{len(to_do)}] {r.file_key} ({human_size(r.size_bytes)}) started {now_iso()}")
            st = process_file(r, TARGET_DOI_SET, test=False, run_id=RUN_ID)
            run_stats.append(st)
            if st.get("records_prefiltered"):
                print(f"   ({st['records_examined'] - st['records_prefiltered']:,} records contained a target DOI prefix "
                      f"and were read in full; references below are counted for those)")
            print(f"   -> {st['status']}: {st['records_examined']:,} records, {st['cited_refs_examined']:,} refs, "
                  f"{st['unique_relationships']:,} relationships, {st['unique_target_dois_matched']:,} target DOIs, "
                  f"{st['seconds']:,}s")
            if st["status"] != "completed":
                warn(f"{r.file_key}: {st.get('error', '')}")
                if STOP_ON_FILE_ERROR:
                    break
            else:
                print(f"   ✅ {r.file_name}: {SAFE_TO_REMOVE_MSG}")
    finally:
        if run_stats:
            display(pd.DataFrame(run_stats)[["file_key", "status", "records_examined", "cited_refs_examined",
                                             "raw_reference_matches", "unique_relationships",
                                             "unique_target_dois_matched", "seconds", "error"]])
        print(f"\nRun time: {(time.time() - t_run) / 60:,.1f} min")
        refresh_manifest()                                   # record full inventories + processing status

## 4. Results and reports — automatic
### 4a. Coverage — which date ranges are done, missing or overlapping
Built from the archive manifest and the processing log. Saved to `coverage_report/` (CSV files plus one Excel workbook).

* **Expected datasets** – analyzed / identified but not processed / possible match only / not found.
* **Expected-range checks** – gaps or overlaps between the ranges you configured.
* **Observed monthly coverage** – from the publication dates of *processed* archives: months no processed archive covers
  (gaps) and months covered by more than one archive (overlaps). A month counts as covered by an archive when it holds at
  least `COVERAGE_MONTH_MIN_SHARE` of that archive's records, so stray early/late records do not create false overlaps.
* **Archives processed more than once** and **duplicate downloads** (same content fingerprint).
* **WoS UTs shared between archives** – exact overlap counts from the hashed UT lists. Shared records never inflate
  citation counts (the master is deduplicated), but they show where datasets overlap.

In [ ]:
def month_range(a, b):
    y, m = int(a[:4]), int(a[5:7])
    out = []
    while f"{y}-{m:02d}" <= b:
        out.append(f"{y}-{m:02d}")
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)
    return out

def compress_months(months):
    if not months:
        return ""
    months, spans, start, prev = sorted(months), [], None, None
    for mo in months:
        if start is None:
            start = prev = mo
        elif month_range(prev, mo)[1:2] == [mo]:
            prev = mo
        else:
            spans.append((start, prev)); start = prev = mo
    spans.append((start, prev))
    return ", ".join(a if a == b else f"{a} to {b}" for a, b in spans)

M = refresh_manifest()
REPORT = {}
print("Archive coverage summary (archives currently present):")
REPORT["archives"] = coverage_summary_table(M)
display(REPORT["archives"])
gone = M[M.currently_present == "no"]
if len(gone):
    print(f"\n{len(gone)} archive(s) seen earlier are no longer present (their results are kept):")
    display(coverage_summary_table(gone, present_only=False))

# ---- 1. Expected datasets ------------------------------------------------------------------
share_in = {}                                     # fingerprint -> {dataset: share of dated records inside}
for r in M.itertuples():
    inv = load_inventory(r.content_fingerprint)
    if inv:
        share_in[r.content_fingerprint] = {nm: i for nm, i, _, _ in range_shares(inv["buckets"])[1]}
rows = []
for e in EXPECTED_DF.itertuples():
    assigned = M[M.dataset_name == e.dataset_name]
    processed = assigned[assigned.times_citation_completed.map(_int) > 0]
    maybe = M[(M.suggested_dataset == e.dataset_name) & (M.dataset_name != e.dataset_name)]
    if len(processed):
        confirmed = processed.dataset_assignment_source.str.startswith("manual").all()
        status = "ANALYZED" + (" (manually confirmed)" if confirmed else " (identified from XML dates - please confirm)")
    elif len(assigned):
        status = "identified, NOT yet processed"
    elif len(maybe):
        status = "possible match only (" + ", ".join(sorted(set(maybe.suggestion_confidence))) + ") - review"
    elif any(v.get(e.dataset_name, 0) >= 0.10 for v in share_in.values()):
        part = [f"{M.loc[M.content_fingerprint == fp, 'current_file_name'].iloc[0]} ({v[e.dataset_name]:.0%})"
                for fp, v in share_in.items() if v.get(e.dataset_name, 0) >= 0.10]
        status = "partly inside other archive(s): " + ", ".join(part) + " - review"
    else:
        status = "NOT FOUND among archives seen so far"
    rows.append({"dataset_name": e.dataset_name, "expected_start": e.start, "expected_end": e.end,
                 "status": status,
                 "archives": MULTI_SEP.join(assigned.current_file_name) or MULTI_SEP.join(maybe.current_file_name),
                 "records_processed": sum(map(_int, processed.record_count)),
                 "expected_records": "" if e.expected_records is None or pd.isna(e.expected_records) else int(e.expected_records),
                 "note": "MORE THAN ONE archive assigned - check for duplicates/splits" if len(assigned) > 1 else ""})
REPORT["expected_datasets"] = pd.DataFrame(rows)
if len(EXPECTED_DF):
    print("\nExpected TDM Studio datasets:")
    display(REPORT["expected_datasets"])
    n_done = REPORT["expected_datasets"].status.str.startswith("ANALYZED").sum()
    print(f"   {n_done} of {len(EXPECTED_DF)} expected date ranges analyzed.")

# ---- 2. Gaps / overlaps between the configured expected ranges ----------------------------------
chk = []
for a, b in zip(EXPECTED_DF.itertuples(), EXPECTED_DF.iloc[1:].itertuples()):
    if b.start > a.end + timedelta(days=1):
        chk.append({"issue": "GAP between expected ranges", "from": a.end + timedelta(days=1),
                    "to": b.start - timedelta(days=1), "datasets": f"{a.dataset_name} / {b.dataset_name}"})
    elif b.start <= a.end:
        chk.append({"issue": "OVERLAP between expected ranges", "from": b.start, "to": min(a.end, b.end),
                    "datasets": f"{a.dataset_name} / {b.dataset_name}"})
REPORT["expected_range_checks"] = pd.DataFrame(chk, columns=["issue", "from", "to", "datasets"])
print("\nConfigured expected ranges:", "contiguous, no gaps or overlaps" if not chk else "")
if chk:
    display(REPORT["expected_range_checks"])

# ---- 3. Observed monthly coverage of PROCESSED archives -------------------------------------------
proc = M[(M.times_citation_completed.map(_int) > 0) & (M.inventory_level == "full")]
cover, records_by_month = defaultdict(list), Counter()
for r in proc.itertuples():
    inv = load_inventory(r.content_fingerprint)
    monthly = Counter()
    for (p_, v), c in inv["buckets"].items():
        if p_ not in ("none", "year"):
            monthly[v[:7]] += c
    tot = sum(monthly.values())
    label = r.dataset_name or r.current_file_name
    for mo, c in monthly.items():
        records_by_month[mo] += c
        if tot and c / tot >= COVERAGE_MONTH_MIN_SHARE:
            cover[mo].append(label)
if cover:
    span = month_range(min(cover), max(cover))
    timeline = pd.DataFrame([{"month": mo, "records_in_processed_archives": records_by_month.get(mo, 0),
                              "n_archives_covering": len(cover.get(mo, [])),
                              "archives_covering": MULTI_SEP.join(sorted(cover.get(mo, [])))} for mo in span])
    gaps = timeline[timeline.n_archives_covering == 0].month.tolist()
    overlaps = timeline[timeline.n_archives_covering > 1].month.tolist()
    print(f"\nObserved coverage of processed archives: {span[0]} to {span[-1]}")
    print("   Gaps (no processed archive covers the month):", compress_months(gaps) or "none")
    print("   Overlaps (covered by 2+ processed archives)  :", compress_months(overlaps) or "none")
    exp_uncovered = []
    for e in EXPECTED_DF.itertuples():
        missing = [mo for mo in month_range(str(e.start)[:7], str(e.end)[:7]) if mo not in cover]
        if missing:
            exp_uncovered.append({"dataset_name": e.dataset_name, "months_not_covered": compress_months(missing)})
    REPORT["expected_months_not_covered"] = pd.DataFrame(exp_uncovered, columns=["dataset_name", "months_not_covered"])
    REPORT["monthly_coverage"] = timeline
    if exp_uncovered:
        print("   Expected-range months with no processed archive yet:")
        display(REPORT["expected_months_not_covered"])
else:
    print("\nNo fully inventoried processed archives yet - observed coverage will appear after the first full run.")

# ---- 4. Archives processed more than once / duplicate downloads ---------------------------------
log = read_log()
comp = log[(log.status == "completed") & (log.content_fingerprint != "")]
twice = comp.groupby("content_fingerprint").agg(times_completed=("run_id", "size"),
                                                file_names=("file_key", lambda s: MULTI_SEP.join(sorted(set(s)))),
                                                runs=("run_id", lambda s: MULTI_SEP.join(s))).reset_index()
twice = twice[twice.times_completed > 1]
REPORT["processed_more_than_once"] = twice
print(f"\nArchives processed for citations more than once: {len(twice)}"
      + (" (expected only if you used FORCE_REPROCESS_FILES; counts are not inflated)" if len(twice) else ""))
if len(twice):
    display(twice)
dup_present = WOS_FILES_DF[WOS_FILES_DF.content_fingerprint.duplicated(keep=False)]
REPORT["duplicate_downloads"] = dup_present[["file_key", "size", "fingerprint_short"]]
print(f"Duplicate downloads currently present (identical content): {len(dup_present)}")
if len(dup_present):
    display(REPORT["duplicate_downloads"])
names = M[M.dataset_name != ""].groupby("dataset_name").content_fingerprint.nunique()
for nm in names[names > 1].index:
    warn(f"Dataset {nm} is assigned to {names[nm]} archives with DIFFERENT content - check the assignments.")

# ---- 5. WoS UTs shared between archives ------------------------------------------------------------
ut_sets = {}
for r in M.itertuples():
    f = ARCHIVE_DIR / f"{r.content_fingerprint[:16]}.uts.npy"
    if f.exists():
        ut_sets[r.content_fingerprint] = (r.dataset_name or r.current_file_name, np.load(f))
pairs = []
keys = list(ut_sets)
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        (na, a), (nb, b) = ut_sets[keys[i]], ut_sets[keys[j]]
        k = int(np.intersect1d(a, b, assume_unique=True).size)
        if k:
            pairs.append({"archive_a": na, "archive_b": nb, "shared_uts": k,
                          "pct_of_a": round(100 * k / max(len(a), 1), 2), "pct_of_b": round(100 * k / max(len(b), 1), 2)})
REPORT["shared_uts_between_archives"] = pd.DataFrame(pairs, columns=["archive_a", "archive_b", "shared_uts",
                                                                     "pct_of_a", "pct_of_b"])
total_unique = len(np.unique(np.concatenate([v for _, v in ut_sets.values()]))) if ut_sets else 0
print(f"\nFully inventoried archives: {len(ut_sets)} | unique WoS UTs across them: {total_unique:,}")
print(f"Archive pairs sharing WoS UTs: {len(pairs)}")
if pairs:
    display(REPORT["shared_uts_between_archives"])

REPORT["manifest"] = M
for name, df in REPORT.items():
    df.to_csv(COVERAGE_DIR / f"{name}.csv", index=False)
write_excel({k[:31]: v for k, v in REPORT.items()}, COVERAGE_DIR / "GREI_WoS_coverage_report.xlsx")
print("\nCoverage report saved in", COVERAGE_DIR)

### 4b. Optional — rename identified exports (off unless `RENAME_ARCHIVES = True`)
Suggests clearer names (e.g. `wos-data (2).zip` → `GREIWOS20230508.zip`). **Nothing is renamed unless
`RENAME_ARCHIVES = True`**, and by default only archives whose dataset you assigned manually are eligible
(`RENAME_REQUIRE_MANUAL`). Existing files are never overwritten, `.part` files are never touched, and every rename is
recorded in `GREI_WoS_archive_rename_log.csv`. Renaming does not trigger reprocessing (archives are tracked by content).

In [ ]:
def rename_plan():
    rows = []
    for r in WOS_FILES_DF.drop_duplicates("content_fingerprint").itertuples():
        m = MANIFEST_LOOKUP.get(r.content_fingerprint, {})
        sug = m.get("suggested_file_name", "")
        if not sug:
            continue
        src = Path(r.path)
        dst = src.with_name(sug)
        basis = m.get("dataset_assignment_source", "")
        if "CONFLICT" in basis:
            reason = "manual assignment conflicts with XML dates - resolve first"
        elif RENAME_REQUIRE_MANUAL and not basis.startswith("manual"):
            reason = "dataset not manually assigned (RENAME_REQUIRE_MANUAL = True)"
        elif dst.exists():
            reason = f"{sug} already exists"
        elif not RENAME_ARCHIVES:
            reason = "RENAME_ARCHIVES = False (suggestion only)"
        else:
            reason = ""
        rows.append({"current_name": r.file_key, "suggested_name": sug, "basis": basis,
                     "will_rename": reason == "", "reason_not_renamed": reason,
                     "src_path": src, "dst_path": dst, "fingerprint": r.content_fingerprint})
    return pd.DataFrame(rows)

PLAN = rename_plan()
if PLAN.empty:
    print("No rename suggestions (no archive identified yet, or names already match their dataset).")
else:
    display(PLAN.drop(columns=["src_path", "dst_path", "fingerprint"]))
    renamed = 0
    for r in PLAN[PLAN.will_rename].itertuples():
        if r.dst_path.exists() or not r.src_path.exists():
            continue
        os.rename(r.src_path, r.dst_path)
        renamed += 1
        append_csv(RENAME_LOG_CSV, [{"renamed_at": now_iso(), "from": str(r.src_path), "to": str(r.dst_path),
                                     "content_fingerprint": r.fingerprint, "basis": r.basis}],
                   ["renamed_at", "from", "to", "content_fingerprint", "basis"])
        i = WOS_FILES_DF.index[WOS_FILES_DF.path == str(r.src_path)]
        WOS_FILES_DF.loc[i, "path"] = str(r.dst_path)
        WOS_FILES_DF.loc[i, "file_name"] = r.dst_path.name
        WOS_FILES_DF.loc[i, "file_key"] = r.dst_path.relative_to(WOS_ROOT).as_posix()
        print(f"Renamed {r.src_path.name} -> {r.dst_path.name}")
    if renamed:
        refresh_manifest()
    elif not RENAME_ARCHIVES:
        print("Nothing renamed. Set RENAME_ARCHIVES = True to apply eligible suggestions.")

### 4c. Citation counts per target DOI, added to every row of your file
Built from the **deduplicated master** (all rounds so far). In test mode, the test edges are used and `TEST_` files are written.

| Column | Definition |
|---|---|
| `wos_citing_documents` | number of unique WoS documents (by `citing_doc_key`) whose reference list cites the DOI |
| `wos_unique_citing_dois` | distinct DOIs among those citing documents |
| `wos_earliest_citing_year` / `wos_latest_citing_year` | range of citing publication years |
| `wos_unique_citing_countries` / `wos_unique_citing_organizations` | distinct countries / organizations across citing documents' addresses |
| `wos_citing_countries` | the list of those countries |

In [ ]:
ANALYSIS_TEST = TEST_MODE
AP = output_paths(ANALYSIS_TEST)
MASTER = build_master(test=ANALYSIS_TEST, write_xlsx=True)

def split_multi(s):
    return [x.strip() for x in str(s).split(MULTI_SEP) if x.strip()] if s else []

AGG_COLS = ["wos_citing_documents", "wos_unique_citing_dois", "wos_earliest_citing_year", "wos_latest_citing_year",
            "wos_unique_citing_countries", "wos_unique_citing_organizations", "wos_citing_countries"]

def aggregate(master):
    if master.empty:
        return pd.DataFrame(columns=["target_doi_normalized"] + AGG_COLS)
    m = master.copy()
    m["_year"] = pd.to_numeric(m.citing_year, errors="coerce")
    rows = []
    for doi, g in m.groupby("target_doi_normalized"):
        countries = sorted({c for v in g.citing_countries for c in split_multi(v)})
        orgs = {o for v in g.citing_organizations for o in split_multi(v)}
        rows.append({"target_doi_normalized": doi,
                     "wos_citing_documents": g.citing_doc_key.nunique(),
                     "wos_unique_citing_dois": g.loc[g.citing_doi != "", "citing_doi"].nunique(),
                     "wos_earliest_citing_year": g._year.min(),
                     "wos_latest_citing_year": g._year.max(),
                     "wos_unique_citing_countries": len(countries),
                     "wos_unique_citing_organizations": len(orgs),
                     "wos_citing_countries": MULTI_SEP.join(countries)})
    agg = pd.DataFrame(rows)
    for c in ["wos_earliest_citing_year", "wos_latest_citing_year"]:
        agg[c] = agg[c].astype("Int64")
    return agg.sort_values("wos_citing_documents", ascending=False).reset_index(drop=True)

AGG = aggregate(MASTER)
atomic_to_csv(AGG, AP["agg"])

MERGED = targets_df.merge(AGG, on="target_doi_normalized", how="left")
assert len(MERGED) == len(targets_df), "merge changed the number of workbook rows"
for c in ["wos_citing_documents", "wos_unique_citing_dois", "wos_unique_citing_countries",
          "wos_unique_citing_organizations"]:
    MERGED[c] = pd.to_numeric(MERGED[c], errors="coerce").fillna(0).astype(int)
MERGED["wos_citing_countries"] = MERGED["wos_citing_countries"].fillna("")
MERGED = MERGED.sort_values("source_excel_row").reset_index(drop=True)

# column order: all ORIGINAL columns first (unchanged), then everything added by this notebook
added = [c for c in MERGED.columns if c not in ORIGINAL_COLUMNS]
MERGED = MERGED[ORIGINAL_COLUMNS + added]

log_now = read_log()
files_done = count_completed()
readme = pd.DataFrame({"item": [
    "Generated", "Mode", "Workbook rows", "Unique target DOIs", "WoS files completed (all rounds)",
    "wos_citing_documents", "attributes_citationCount", "Matching", "Multi-value separator"],
    "value": [now_iso(), "TEST (sample only)" if ANALYSIS_TEST else "full", len(MERGED), len(TARGET_DOI_SET),
              files_done,
              "Unique Web of Science documents whose cited-reference list contains the target DOI (exact match after "
              "normalization of the 'id' column), across the WoS files processed so far.",
              "Original DataCite value, unchanged; a different citation source.",
              "Exact normalized DOI match in WoS cited references; never the citing record's own DOI.",
              MULTI_SEP]})

atomic_to_csv(MERGED, AP["merged_csv"])
write_excel({"targets_with_wos": MERGED, "about": readme}, AP["merged_xlsx"])
print(f"Targets with ≥1 WoS citing document: {int((AGG.wos_citing_documents > 0).sum()) if len(AGG) else 0:,}")
print("Saved:", AP["agg"].name, "|", AP["merged_xlsx"].name, "|", AP["merged_csv"].name)
display(AGG.head(10))

### 4d. DataCite `attributes_citationCount` vs WoS `wos_citing_documents`
These are **different citation universes** (DataCite/Crossref event data vs. WoS cited references within the processed
WoS years). Differences are expected and are **not** errors. One row per unique target DOI.

In [ ]:
cmp = MERGED[MERGED.target_doi_normalized.notna()].drop_duplicates("target_doi_normalized").copy()
cmp["datacite_citationCount"] = pd.to_numeric(cmp.get("attributes_citationCount"), errors="coerce")
cmp["difference_wos_minus_datacite"] = cmp.wos_citing_documents - cmp.datacite_citationCount
def _cat(r):
    d, w = r.datacite_citationCount, r.wos_citing_documents
    if pd.isna(d):
        return "DataCite count missing"
    if d == 0 and w == 0: return "both zero"
    if d > 0 and w == 0: return "DataCite only"
    if d == 0 and w > 0: return "WoS only"
    return "both > 0"
cmp["comparison_category"] = cmp.apply(_cat, axis=1)
cmp_cols = [c for c in [TARGET_DOI_COLUMN, "target_doi_normalized", "target_title", "attributes_publicationYear",
                        "attributes_types_resourceTypeGeneral", "relationships_client_data_id",
                        "datacite_citationCount", "wos_citing_documents", "difference_wos_minus_datacite",
                        "comparison_category"] if c in cmp.columns]
COMPARE = cmp[cmp_cols].rename(columns={TARGET_DOI_COLUMN: "target_doi"}).reset_index(drop=True)
atomic_to_csv(COMPARE, AP["compare"])
print("Counts by category (unique DOIs):")
display(COMPARE.comparison_category.value_counts().rename_axis("category").reset_index(name="target_dois"))
both = COMPARE.dropna(subset=["datacite_citationCount"])
if len(both) > 2 and both.wos_citing_documents.nunique() > 1 and both.datacite_citationCount.nunique() > 1:
    print(f"Spearman correlation (DataCite vs WoS): "
          f"{both.datacite_citationCount.corr(both.wos_citing_documents, method='spearman'):.2f}")
print("\nLargest differences:")
display(COMPARE.reindex(COMPARE.difference_wos_minus_datacite.abs().sort_values(ascending=False).index).head(15))
print("Saved:", AP["compare"].name)

### 4e. Summary tables

In [ ]:
uniq_t = MERGED[MERGED.target_doi_normalized.notna()].drop_duplicates("target_doi_normalized")
cited = uniq_t[uniq_t.wos_citing_documents > 0]
print("=" * 70)
print("GREI / DataCite outputs cited in Web of Science" + ("   [TEST SAMPLE ONLY]" if ANALYSIS_TEST else ""))
print("=" * 70)
print(f"Total workbook rows                                : {len(MERGED):,}")
print(f"Unique target DOIs                                 : {len(uniq_t):,}")
print(f"Target DOIs with ≥1 WoS citing document            : {len(cited):,}")
print(f"Share of target DOIs with a WoS citation           : {100 * len(cited) / max(len(uniq_t), 1):.2f}%")
print(f"Unique target DOI × citing-document relationships  : {len(MASTER):,}")
print(f"Unique citing WoS documents                        : {MASTER.citing_doc_key.nunique() if len(MASTER) else 0:,}")
print(f"wos_citing_documents  max / median / mean (all unique targets): "
      f"{uniq_t.wos_citing_documents.max():,} / {uniq_t.wos_citing_documents.median():g} / {uniq_t.wos_citing_documents.mean():.3f}")
if len(cited):
    print(f"wos_citing_documents  median / mean among cited targets       : "
          f"{cited.wos_citing_documents.median():g} / {cited.wos_citing_documents.mean():.2f}")
print(f"WoS files completed so far (all rounds)            : {count_completed():,} (archives currently visible: {WOS_FILES_DF.content_fingerprint.nunique():,})")

top_cols = [c for c in [TARGET_DOI_COLUMN, "target_title", "attributes_publicationYear",
                        "attributes_types_resourceTypeGeneral", "relationships_client_data_id",
                        "attributes_citationCount", "wos_citing_documents", "wos_earliest_citing_year",
                        "wos_latest_citing_year"] if c in uniq_t.columns]
TOP25 = uniq_t.sort_values("wos_citing_documents", ascending=False)[top_cols].head(25).reset_index(drop=True)
TOP25.index += 1
print("\nTop 25 target outputs by wos_citing_documents:")
display(TOP25)

pre = AP["summary_prefix"]
tables = {"top25_targets": TOP25}
if len(MASTER):
    docs = MASTER.drop_duplicates("citing_doc_key")            # one row per citing document
    def explode_count(df, col, label):
        s = df[col].map(split_multi).explode().dropna()
        s = s[s != ""]
        return s.value_counts().rename_axis(label).reset_index(name="citing_documents")
    by_year = MASTER.groupby("citing_year").agg(relationships=("relationship_key", "size"),
                                                citing_documents=("citing_doc_key", "nunique"),
                                                target_dois=("target_doi_normalized", "nunique")).reset_index()
    tables.update({
        "by_citing_year": by_year,
        "top_citing_countries": explode_count(docs, "citing_countries", "country"),
        "top_citing_organizations": explode_count(docs, "citing_organizations", "organization"),
        "top_citing_sources": docs.citing_source.replace("", "(missing)").value_counts()
                                  .rename_axis("source").reset_index(name="citing_documents"),
        "citing_document_types": explode_count(docs, "citing_document_type", "document_type"),
    })
    if "attributes_types_resourceTypeGeneral" in MASTER.columns:
        tables["by_target_resource_type"] = (MASTER.fillna({"attributes_types_resourceTypeGeneral": "(missing)"})
            .groupby("attributes_types_resourceTypeGeneral")
            .agg(relationships=("relationship_key", "size"), target_dois=("target_doi_normalized", "nunique"))
            .reset_index().sort_values("relationships", ascending=False))
    for name in ["by_citing_year", "top_citing_countries", "top_citing_organizations", "top_citing_sources",
                 "citing_document_types"]:
        print(f"\n{name.replace('_', ' ').title()}:")
        display(tables[name].head(25))
for name, t in tables.items():
    if len(t):
        t.to_csv(SUMMARY_DIR / f"{pre}{name}.csv", index=False)
write_excel({k[:31]: v for k, v in tables.items() if len(v)}, SUMMARY_DIR / f"{pre}GREI_WoS_summary_tables.xlsx")
print("\nSummary tables saved in", SUMMARY_DIR)

### 4f. Quality checks
Each check prints `OK` or a `WARNING` with examples. Nothing questionable is silently dropped.

In [ ]:
def qa(label, n, detail_df=None, ok_when_zero=True, show=8):
    flag = "OK " if (n == 0) == ok_when_zero else "⚠️ "
    print(f"{flag} {label}: {n:,}")
    if detail_df is not None and n and flag.startswith("⚠️"):
        display(detail_df.head(show))

print("---- Workbook ----")
dup_ids = targets_df[targets_df[TARGET_DOI_COLUMN].notna() & targets_df[TARGET_DOI_COLUMN].duplicated(keep=False)]
qa("Rows with duplicate raw 'id' values (kept; counted once per DOI)", len(dup_ids),
   dup_ids[["source_excel_row", TARGET_DOI_COLUMN, "relationships_client_data_id"]
           if "relationships_client_data_id" in dup_ids else ["source_excel_row", TARGET_DOI_COLUMN]]
   .sort_values(TARGET_DOI_COLUMN))
bad_t = targets_df[targets_df.target_doi_normalized.isna()]
qa("Malformed / unusable target DOIs", len(bad_t), bad_t[["source_excel_row", TARGET_DOI_COLUMN,
                                                          "target_doi_normalization_status"]])
if HAVE_QA_COL:
    d = targets_df[targets_df.id_vs_attributes_doi == "DIFFER"]
    qa("Rows where normalized id ≠ normalized attributes_doi", len(d),
       d[["source_excel_row", TARGET_DOI_COLUMN, QA_DOI_COLUMN]])

print("\n---- Edges ----")
raw_path = AP["edges_raw"]
RAW = pd.read_csv(raw_path, dtype=str, keep_default_na=False) if raw_path.exists() else pd.DataFrame(columns=EDGE_COLUMNS)
qa("Duplicate relationship keys in the MASTER (must be 0)",
   int(MASTER.relationship_key.duplicated().sum()) if len(MASTER) else 0)
print(f"ℹ️  Duplicate rows in the raw append-only journal (removed in master): {len(RAW) - RAW.relationship_key.nunique():,}"
      + ("" if raw_path.exists() else " (journal not present - fine; the master holds all results)"))
if len(MASTER):
    multi = MASTER[MASTER.n_source_files_with_relationship.astype(int) > 1]
    qa("Relationships found in more than one WoS file (overlap; counted once)", len(multi),
       multi[["target_doi_normalized", "citing_doc_key", "all_source_wos_files"]], ok_when_zero=True)
    ut_files = MASTER[MASTER.citing_wos_ut != ""].groupby("citing_wos_ut").agg(
        files=("all_source_wos_files", lambda col: MULTI_SEP.join(sorted(set().union(*map(_split_multi_set, col)))))
    ).reset_index()
    ut_files["n_files"] = ut_files.files.map(lambda v: len(_split_multi_set(v)))
    ov = ut_files[ut_files.n_files > 1]
    qa("Citing WoS records (UT) seen in more than one WoS archive", len(ov), ov)
    if len(ov):
        print("   overlapping archive combinations:", dict(Counter(ov.files).most_common(5)))
    noUT = MASTER[MASTER.citing_key_type != "UT"]
    qa("Relationships whose citing record has no WoS UT (fallback key used)", len(noUT),
       noUT[["target_doi_normalized", "citing_doc_key", "citing_title", "source_wos_file"]])
    noDOI = MASTER.drop_duplicates("citing_doc_key")
    print(f"ℹ️  Citing documents without their own DOI: {(noDOI.citing_doi == '').sum():,} of {len(noDOI):,} "
          "(normal for some document types; does not affect matching)")
    print(f"ℹ️  Match method: {dict(Counter(MASTER.match_method))}")
    multi_m = MASTER[MASTER.n_mentions_in_reference_list.astype(int) > 1]
    print(f"ℹ️  Relationships where the target appears more than once in one reference list (counted once): {len(multi_m):,}")

mal_p = AP["malformed"]
if mal_p.exists():
    mal = pd.read_csv(mal_p, dtype=str, keep_default_na=False)
    qa("Malformed cited-reference DOI values (samples saved)", len(mal), mal)
near_p = AP["nearmiss"]
if near_p.exists():
    near = pd.read_csv(near_p, dtype=str, keep_default_na=False).drop_duplicates(["citing_wos_ut", "cited_doi", "target_doi"])
    qa("Possible version near-misses (cited DOI differs only by a version suffix; NOT counted — review)", len(near),
       near[["cited_doi", "target_doi", "citing_wos_ut", "reference"]])

unmatched = MERGED[MERGED.target_doi_normalized.notna() & (MERGED.wos_citing_documents == 0)] \
    .drop_duplicates("target_doi_normalized")
keep = [c for c in ["source_excel_row", TARGET_DOI_COLUMN, "target_doi_normalized", "target_title",
                    "attributes_publicationYear", "relationships_client_data_id", "attributes_citationCount"]
        if c in unmatched.columns]
atomic_to_csv(unmatched[keep], AP["unmatched"])
print(f"\nℹ️  Target DOIs with no WoS citing document so far: {len(unmatched):,} (list saved to {AP['unmatched'].name})")

print("\n---- Processing log ----")
log = read_log()
if log.empty:
    print("No processing log yet (normal in test mode).")
else:
    log = log.reset_index().sort_values("index")
    log["_id"] = [fp if fp else "KEY:" + k for fp, k in zip(log.content_fingerprint, log.file_key)]
    last = log.groupby("_id").tail(1)
    print("Latest status per file/archive:", dict(Counter(last.status)))
    problems = last[last.status != "completed"]
    qa("Files whose latest status is not 'completed' (will be retried next run)", len(problems),
       problems[["file_key", "status", "started_at", "error"]])
    fps, legacy = completed_status()
    pending = WOS_FILES_DF.drop_duplicates("content_fingerprint")
    pending = pending[~pending.apply(lambda r: is_completed(r, fps, legacy), axis=1)] if len(pending) else pending
    print(f"ℹ️  Visible WoS archives not yet completed: {len(pending):,}"
          + (f" ({', '.join(pending.file_name.head(5))})" if len(pending) else ""))
    print(f"ℹ️  Incomplete transfers present (never processed): {len(INCOMPLETE_FILES)}")
    done = last[last.status == "completed"].copy()
    for c in ["records_examined", "cited_refs_examined", "cited_refs_with_doi", "malformed_ref_dois"]:
        done[c] = pd.to_numeric(done[c], errors="coerce")
    print(f"ℹ️  Totals over completed files: {done.records_examined.sum():,.0f} records, "
          f"{done.cited_refs_examined.sum():,.0f} cited references "
          f"({100 * done.cited_refs_with_doi.sum() / max(done.cited_refs_examined.sum(), 1):.1f}% with a DOI), "
          f"{done.malformed_ref_dois.sum():,.0f} malformed reference DOIs")
    zero_refs = done[done.cited_refs_examined == 0]
    qa("Completed files in which no cited references were found (schema problem?)", len(zero_refs),
       zero_refs[["file_key", "records_examined"]])

print("\nℹ️  Archive identification, date coverage, duplicate archives and WoS UTs shared between archives: "
      "see section 4a and coverage_report/.")

## 5. ✅ What happened and what to do next

In [ ]:
real_master_path = output_paths(False)["master_csv"]
REAL_MASTER = pd.read_csv(real_master_path, dtype=str, keep_default_na=False) if real_master_path.exists() else pd.DataFrame()
fps_done, legacy_done = completed_status()
present = WOS_FILES_DF.drop_duplicates("content_fingerprint")
this_run = pd.DataFrame(RUN_STATS)
done_now = this_run[this_run.status == "completed"] if len(this_run) else this_run
bad_now = this_run[this_run.status != "completed"] if len(this_run) else this_run
outstanding = present[~present.apply(lambda r: is_completed(r, fps_done, legacy_done), axis=1)] if len(present) else present

print("=" * 78)
print(f"RUN SUMMARY  ({now_iso()})" + ("   [TEST MODE - nothing was marked processed]" if TEST_MODE else f"   [{RUN_ID}]"))
print("=" * 78)
print(f"Archives processed this run               : {len(done_now):,}"
      + (f"  ({', '.join(done_now.file_name)})" if len(done_now) else ""))
if len(bad_now):
    print(f"Archives that FAILED/stopped this run      : {len(bad_now):,}  ({', '.join(bad_now.file_name)}) - will be retried")
print(f"Archives present but completed previously : {len(RUN_SKIPPED):,}")
print(f"Archives still outstanding (present)      : {len(outstanding):,}"
      + (f"  ({', '.join(outstanding.file_name)})" if len(outstanding) else ""))
print(f"Incomplete transfers (.part) waiting      : {len(INCOMPLETE_FILES):,}")
if len(EXPECTED_DF):
    END_EXPECTED = expected_status_table()
    pend = END_EXPECTED[~END_EXPECTED.status.str.startswith("processed")]
    print(f"Expected date ranges not yet processed    : {len(pend):,} of {len(END_EXPECTED):,}"
          + (f"  ({', '.join(pend.dataset_name)})" if len(pend) else ""))
print(f"Archives completed across all runs        : {count_completed():,}")
print(f"Citation relationships accumulated (all runs): {len(REAL_MASTER):,}")
print(f"Unique target DOIs with ≥1 WoS citing document: "
      f"{REAL_MASTER.target_doi_normalized.nunique() if len(REAL_MASTER) else 0:,} of {len(TARGET_DOI_SET):,}")
print(f"Unique citing WoS documents (all runs)    : {REAL_MASTER.citing_doc_key.nunique() if len(REAL_MASTER) else 0:,}")

removable = present[present.apply(lambda r: is_completed(r, fps_done, legacy_done), axis=1)] if len(present) else present

print("\n" + "=" * 78)
print("WHAT TO DO NEXT")
print("=" * 78)
if TEST_MODE:
    if WOS_FILES_DF.empty:
        print(f"• No export found. Download wos-data.zip from TDM Studio into {WOS_ROOT}, then Run All again.")
    elif TEST_OK:
        print("• ✅ The test passed: references were found and matching works.")
        print("• Next: in section 1 set  TEST_MODE = False , then Run All. The full export will be processed")
        print("  (it can take a while for ~1 million records - progress is shown in section 3b).")
    else:
        print("• ❌ The test did not pass. Do NOT start the full run yet.")
        print("  Save this notebook (with its output) and send it to me so we can see what the records look like.")
else:
    if not STATE_OK and not IGNORE_STATE_WARNINGS:
        print("• ❌ Saved results look incomplete (see section 2c). Run  restore_latest_backup()  in a new cell, then Run All.")
    elif len(bad_now):
        print(f"• ⚠️ {', '.join(bad_now.file_name)} did not finish. Just Run All again: it restarts that export.")
        print("  If it fails twice, send me the notebook with its output.")
    elif len(done_now):
        print(f"• ✅ Done: {', '.join(done_now.file_name)} was processed and the results are saved in {OUTPUT_DIR.name}/.")
        print(f"• Next: delete wos-data.zip from {WOS_ROOT}, download the next export from TDM Studio, then Run All.")
    elif WOS_FILES_DF.empty:
        print(f"• No export in {WOS_ROOT}. Download the next wos-data.zip from TDM Studio, then Run All.")
    else:
        print("• Nothing new: the export(s) present were already processed. Delete them and download the next one.")
    print(f"• Results so far: {len(REAL_MASTER):,} citation relationships; "
          f"{REAL_MASTER.target_doi_normalized.nunique() if len(REAL_MASTER) else 0:,} of {len(TARGET_DOI_SET):,} "
          "target DOIs cited at least once.")
    print(f"• Main file to open: {output_paths(False)['merged_xlsx'].name} (and the citation list "
          f"{output_paths(False)['master_xlsx'].name}).")
if len(removable) and not TEST_MODE:
    print("\nSafe to remove from the workspace (processed successfully; results saved):")
    for r in removable.itertuples():
        print(f"   🗑  {r.file_key} ({r.size})")
    dups = WOS_FILES_DF[WOS_FILES_DF.content_fingerprint.isin(removable.content_fingerprint)
                        & ~WOS_FILES_DF.file_key.isin(removable.file_key)]
    for r in dups.itertuples():
        print(f"   🗑  {r.file_key} ({r.size}) - duplicate download of a processed archive")
    print("   The notebook never deletes source files. Keep everything in", OUTPUT_DIR.name + "/",
          "(log, master, manifest, archive_inventory/).")

## Reference: how it works, definitions and output files

> This is a **cited-reference** analysis: target DOIs are matched against the DOIs *inside each WoS record's reference list* — never against the DOI of the WoS record itself.

> **Disk use.** ZIPs are read in place: XML members are decompressed one at a time in memory and parsed as a stream —
> nothing is extracted to disk and no temporary copy of a ZIP is ever made. New files are only the small outputs in
> `OUTPUT_DIR` (for a ~1 M-record archive the hashed UT list is ~9 MB). Budget free disk space for one export plus a few hundred MB
> for outputs, and at least 4 GB of free memory (opening a ZIP with ~1 million members takes ~0.7 GB while it is read).
>
> **Storage model.** The ~10 GB WoS ZIPs are *temporary inputs*. The persistent state is a handful of small files in
> `OUTPUT_DIR` (processing log, citation-edge master, archive manifest, merged workbook, `archive_inventory/`).
> Never delete `OUTPUT_DIR`; delete processed ZIPs freely. The *Run status* table (section 2c) and section 5 show what is new, done, missing or still downloading.

### Key definitions
* **Citation relationship** = one target DOI × one citing WoS document. If a document cites the same DOI several times in its
  reference list, that is still **one** relationship (`n_mentions_in_reference_list` records how many times).
* **`wos_citing_documents`** = number of **unique WoS documents** found citing the target DOI *in the WoS files processed so far*.
  It is deliberately **not** called "total citations".
* **`attributes_citationCount`** (from DataCite) is kept unchanged and is a different citation universe.

### Output files (all in `OUTPUT_DIR`)
| File | Contents |
|---|---|
| `GREI_WoS_processing_log.csv` | Append-only audit log, one or more rows per WoS input file per run |
| `GREI_WoS_citation_edges_raw_appendonly.csv` | Every match ever written (may contain duplicates from overlapping datasets / reruns) |
| `GREI_WoS_citation_edges_master.csv` / `.xlsx` | **Deduplicated** citation-edge master (one row per target DOI × citing document) |
| `GREI_WoS_target_aggregates.csv` | Per-target-DOI WoS metrics |
| `GREI-DataCite-NIHOnly_with_WoS_citations.xlsx` / `.csv` | Original workbook + WoS metrics on every row |
| `GREI_WoS_vs_DataCite_citation_comparison.csv` | DataCite count vs WoS-derived count |
| `GREI_WoS_unmatched_target_dois.csv` | Target DOIs with no WoS citing document yet |
| `GREI_WoS_malformed_cited_ref_doi_samples.csv` | Samples of DOI-like strings in references that could not be normalized |
| `GREI_WoS_possible_version_near_misses.csv` | Cited DOIs that differ from a target only by a version suffix (**reported, never counted**) |
| `summary_tables/` | Summary tables from section 4e |
| `GREI_WoS_archive_manifest.csv` | One row per WoS archive ever seen: content fingerprint, current name, record/UT counts, publication-date range and distribution, suggested and assigned TDM Studio dataset, processing status. The `manual_*` and `notes` columns are yours to edit |
| `coverage_report/` | Which expected date ranges are analyzed / missing, gaps, overlaps, duplicate archives, UTs shared between archives |
| `archive_inventory/` | Per-archive date profiles and hashed UT lists used by the manifest and coverage report |
| `GREI_WoS_archive_rename_log.csv` | Record of any renames (only if `RENAME_ARCHIVES = True`) |

Files produced in test mode carry a `TEST_` prefix and never touch the real log or master.

## Reference: rolling-batch routine (each new export)
1. In TDM Studio, export the next chunk; it downloads to your Downloads folder as `wos-data.zip`. Wait until the browser
   has finished (no `.crdownload` / `.part` / `.download` file left — the start-of-run table lists any still downloading).
   Do not unzip it.
2. Keep `GREI_WoS_outputs/` (next to this notebook) and its files; `TEST_MODE = False`.
3. *Kernel → Restart & Run All.* The start-of-run table shows the export as **NEW**, **previously processed** or
   **will be retried**, plus earlier exports that are no longer present and expected date ranges still missing.
4. Only new (or previously failed) exports are processed. When one finishes, its matches are merged into the master
   **before** the log marks it completed, and the notebook prints that it can be deleted.
5. The aggregates, merged results file, summaries, coverage report and QA are rebuilt from the **whole accumulated master**.
6. Delete `wos-data.zip` (the notebook never deletes it) and repeat with the next export. Re-using the same file name is
   fine: each export is recognised by its content, not its name.

**What must be kept between runs:** everything in `OUTPUT_DIR`, the target CSV and this notebook. The
`*_raw_appendonly.csv` journal is optional once a run has finished (its rows are already in the master). Each full run
first copies the small state files to `OUTPUT_DIR/state_backups/<run>` (last `KEEP_STATE_BACKUPS` kept); restore from there
if a state file is ever damaged or deleted. A start-of-run integrity check warns if any completed archive's matches are
missing from the master.

**Generic export names.** Each archive is tracked by a content fingerprint, so: a renamed archive is not reprocessed;
an identical re-download (`wos-data (4).zip` = `wos-data (1).zip`) is detected and processed only once; and a *new* export
that reuses an old generic name is recognized as new. Files still transferring (`*.part`) are listed but never touched —
re-run once the download finishes. After each round, check section 4a to see which expected date ranges are still missing,
and record confirmed identities in `ARCHIVE_ASSIGNMENTS` so the provenance in the manifest and edge files is explicit.

**Archives processed with an older version of this notebook** have blank values in the newer metadata columns. To fill
them, download that export again and add it to `FORCE_REPROCESS_FILES`: existing values are kept and blanks are filled from the
new pass; relationships are still counted once.

**To reprocess a file deliberately:** add its name or a pattern to `FORCE_REPROCESS_FILES`. Its new matches are deduplicated
against old ones automatically.

**To start completely fresh:** move the whole `OUTPUT_DIR` aside (keep it as an audit trail) and run again.

**Notes on interpretation**
* Only references for which WoS recorded a DOI (or that contain a DOI in their text, if
  `SCAN_ALL_REFERENCE_TEXT_FOR_DOIS = True`) can match. Data citations without a DOI in the WoS reference are not found.
* Version near-misses (e.g., `…figshare.123.v2` vs `…figshare.123`) are listed for review but never counted, keeping
  matching strictly exact.
* `wos_citing_documents` covers only the WoS files processed so far; record which datasets/years were included when
  reporting.